# Importar pacotes

In [ ]:
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess,Fourier
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.neural_network import MLPRegressor

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

# Métodos utilitários

In [ ]:
from utils.common import import_dataframe
from utils.common import make_lags, make_leads
from utils.common import calculate_metrics
from utils.common import HybridRecursive, HybridBase
from utils.common import ModeloPrevisaoVolume

# Importação dos dados

In [ ]:
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df.set_index("Data", inplace=True)
df = df[df.index >= "2018-01-01"]


# Construção - Vazão Natural  e Juzante

In [ ]:

VALIDATION_SIZE = 1*90

def vazao_dados(df, nome_coluna_vazao):
  y_vazao = df[nome_coluna_vazao]
  fourier = CalendarFourier(freq="YE", order=2)
  y_vazao = y_vazao.asfreq("D")
  dp = DeterministicProcess(
    index=y_vazao.index,
    constant=False,
    order=1,
    seasonal=False,
    additional_terms=[fourier],
    drop=True,
  )
  
  X_full_vazao = dp.in_sample()
  return X_full_vazao, y_vazao

X_j, y_j = vazao_dados(df, nome_coluna_vazao_jusante)
X_n, y_n = vazao_dados(df, nome_coluna_vazao_natural)
df_aux = pd.concat([pd.DataFrame(X_j).add_suffix('_jusante'),
                    pd.DataFrame(X_n).add_suffix('_natural'),
                    y_n, y_j], axis=1)


In [ ]:
#### DENTRO DO FOR PORQUE FAZ AUTOREGRESSÃO -> SO VALE PARA TREINO. 
def volume_dados(df, y_vn, y_vj, TARGET_COLUMN_NAME = 'Volume Útil Armazenado (%)'):
  y_vol = df[TARGET_COLUMN_NAME]
  X_lags = make_lags(y_vol.squeeze(), 1)
  X_Qin_leads = make_leads(y_vn.squeeze(), 1, name="Qin")
  X_Qout_leads = make_leads(y_vj.squeeze(), 1, name="Qout")
  X_full_vol = pd.concat([X_lags, X_Qin_leads, X_Qout_leads], axis=1).dropna()
  y_vol, X_full_vol = y_vol.align(X_full_vol, join='inner', axis=0)
  return X_full_vol, y_vol


## Treino

In [ ]:
N_SPLITS = 5
FORECAST_HORIZON = VALIDATION_SIZE
tscv = TimeSeriesSplit(
    n_splits=N_SPLITS,
    test_size=FORECAST_HORIZON,
    gap=0 # Sem gap entre treino e teste
)

all_preds_model = []
all_y_test = []
lista_modelos_v = [LinearRegression(fit_intercept=False)]
lista_modelos_j = [HybridRecursive(LinearRegression(), RandomForestRegressor(), lags=2) for i in range(0, len(lista_modelos_v))]
lista_modelos_n = [HybridRecursive(Lasso(), KNeighborsRegressor(), lags=2) for i in range(0, len(lista_modelos_v))]



#### FOR PARA CADA MODELO A SER TESTADO ###
for pos, modelo_v in enumerate(lista_modelos_v):

    model = ModeloPrevisaoVolume(modelo_v, usar_jusante=True,
                                modelo_vazao_juzante = lista_modelos_j[pos],
                                modelo_vazao_natural = lista_modelos_n[pos])


#### DENTRO DE UM FOR PARA CADA SPLIT DO TIME SERIES SPLIT ####
    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
        df_train = df.iloc[train_idx]
        df_test = df.iloc[test_idx]
        df_aux_train = df_aux.iloc[train_idx]
        df_aux_test = df_aux.iloc[test_idx]        
        X_full_vol_train, y_vol_train = volume_dados(df_train, y_n, y_j)

        model.fit_vazao_jusante(df_aux_train.filter(like='jusante'), y_j.loc[df_aux_train.index])
        model.calculate_lags(df_aux_test.filter(like='jusante'))
        y_pred_train_vj = model.predict(df_aux_train.filter(like='jusante'))
        y_pred_test_vj = model.predict(df_aux_test.filter(like='jusante'))

        model.fit_vazao_natural(df_aux_test.filter(like='natural'), y_n.loc[df_aux_train.index])
        model.calculate_lags(df_aux_test.filter(like='natural'))
        y_pred_train_vn = model.predict(df_aux_train.filter(like='natural'))
        y_pred_test_vn = model.predict(df_aux_test.filter(like='natural'))
        
        model.fit(X_full_vol_train, y_vol_train)

        #### VAI FAZER O CALCULO E SALVAR A METRICA ####
        
        
        

PARA DESACOPLAR

Usar modelos separados.

1) Criar função Calculate lags independente de modelo ou de qualquer coisa (SEM CLASSE).
2) Fazer para cada tipo (VAZAO PRIMEIRO) o fit do modelo 1 (que considera sazonalidade e tendencia)
3) Fazer o predict deste modelo
4) Fazer yreal - y_pred (para calcular o ciclo Lag2), pegar o modelo 2 e fazer fit dele considerando lag do residuo.
5) Obter o valor predito final y_pred_final -> somando y_pred + y_pred_residuo
6) Pegar o y_pred_final (da vazao) e unir com o y_pred_final da outra vazao e usar o lag do volume como variáveis do modelo de volume que faz fit e predict.
OBS: O MODELO É RECURSIVO

In [ ]:
def calculate_lags_treino(df_aux, column, lags = 1,):
    df_aux_lags = df_aux[column].shift(lags).rename(f"{column}_lag{lags}")
    df_aux_lags = df_aux_lags.fillna(method='ffill').fillna(method='bfill')
    df_ = pd.concat([df_aux, df_aux_lags], axis=1)
    return df_




In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.metrics import mean_squared_error
import warnings
import matplotlib.pyplot as plt

# Ignore warnings that match the specified criteria
warnings.filterwarnings('ignore', message='.*deprecated.*', category=DeprecationWarning)

# Issue a warning
warnings.warn('This is a deprecated feature', DeprecationWarning)

# --- 1. CONFIGURAÇÃO E DADOS ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df[(df['Data']<='2014-03-01')|(df['Data']>='2016-02-15')]
df.set_index("Data", inplace=True)
#(k['Data']<='2014-03-01')|(k['Data']>='2016-02-15')

#df = df[df.index >= "2018-01-01"]

VALIDATION_SIZE = 90
LAGS_VAZAO = 2
LAGS_VOLUME = 1

# --- 2. FUNÇÕES AUXILIARES (DESACOPLADAS) ---

def criar_features_deterministicas(index, order=2):
    """Cria X para sazonalidade e tendência (usado pelos modelos lineares de vazão)."""
    fourier = CalendarFourier(freq="YE", order=order)
    dp = DeterministicProcess(
        index=index,
        constant=True,
        order=1, # Tendência linear
        seasonal=False,
        additional_terms=[fourier],
        drop=True,
    )
    return dp.in_sample()

def criar_features_deterministicas_volume(index):
    dp = DeterministicProcess(
        index=index,
        constant=True, order=1,
        additional_terms=[
            Fourier(period=365.25 * 2, order=1),
            CalendarFourier(freq="YE", order=2)
        ],
        drop=True
    )
    return dp.in_sample()

def criar_lags(series, lags):
    """Cria DataFrame com lags para treino."""
    df_lags = pd.DataFrame(index=series.index)
    for lag in range(1, lags + 1):
        df_lags[f'lag_{lag}'] = series.shift(lag)
    return df_lags

def treinar_modelo_hibrido_vazao(y_train, modelo_trend, modelo_resid, lags):
    """
    Treina o combo: Modelo Linear (Tendência) + Modelo ML (Resíduos Lags).
    Retorna os modelos treinados e os últimos resíduos para iniciar o loop de teste.
    """
    # 1. Features Determinísticas
    X_trend = criar_features_deterministicas(y_train.index)
    
    # 2. Fit Modelo Tendência
    modelo_trend.fit(X_trend, y_train)
    y_pred_trend = modelo_trend.predict(X_trend)
    y_pred_trend = pd.Series(y_pred_trend, index=y_train.index)
    
    # 3. Calcular Resíduos
    residuos = y_train - y_pred_trend
    
    # 4. Criar Lags dos Resíduos para o Modelo 2
    X_resid = criar_lags(residuos, lags).dropna()
    y_resid_target = residuos.loc[X_resid.index]
    
    # 5. Fit Modelo Resíduo
    modelo_resid.fit(X_resid, y_resid_target)
    
    # Retorna modelos e os ultimos dados de residuos para servir de 'semente' no teste
    last_residuals = residuos.tail(lags).values
    
    return modelo_trend, modelo_resid, last_residuals

def treinar_modelo_volume(df_train, col_vol, col_vn, col_vj, model_vol):
    """
    Treina o modelo de volume usando:
    Target: Volume(t)
    Features: Volume(t-1), Vn(t), Vj(t)
    Nota: No treino usamos as vazões REAIS (Teacher Forcing).
    """
    y = df_train[col_vol]
    
    # Features
    X_dynamic = pd.DataFrame(index=df_train.index)
    X_dynamic['vol_lag1'] = df_train[col_vol].shift(1)
    X_dynamic['vazao_n'] = df_train[col_vn] # Vazão no dia atual (conforme pedido)
    X_dynamic['vazao_j'] = df_train[col_vj] # Vazão no dia atual
    

    X_trend = criar_features_deterministicas_volume(df_train.index)
    
    # Junta tudo (alinha pelo índice automaticamente)
    X_full = pd.concat([X_dynamic, X_trend], axis=1).dropna()
    
    # Alinha y com X (por causa dos NaNs do lag)
    y_train = y.loc[X_full.index]
    
    model_vol.fit(X_full, y_train)
    return model_vol



# --- 3. LOOP DE VALIDAÇÃO (TIME SERIES SPLIT) ---

tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)

resultados = []

print("Iniciando Validação Cruzada...\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
    # Separação dos dados
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]
    
    print(f"Fold {fold+1}: Treino até {df_train.index[-1].date()} | Teste de {df_test.index[0].date()} a {df_test.index[-1].date()}")

    # =================================================================
    # ETAPA 1: TREINAMENTO (Usando apenas dados de Treino)
    # =================================================================
    
    # --- A. Treinar Vazão Natural (Híbrido) ---
    model_trend_n = Lasso(alpha=0.1)
    model_resid_n = KNeighborsRegressor(n_neighbors=5)
    
    m_trend_n, m_resid_n, last_resids_n = treinar_modelo_hibrido_vazao(
        df_train[nome_coluna_vazao_natural], model_trend_n, model_resid_n, LAGS_VAZAO
    )
    
    # --- B. Treinar Vazão Jusante (Híbrido) ---
    model_trend_j = LinearRegression()
    model_resid_j = RandomForestRegressor(n_estimators=50, random_state=42)
    
    m_trend_j, m_resid_j, last_resids_j = treinar_modelo_hibrido_vazao(
        df_train[nome_coluna_vazao_jusante], model_trend_j, model_resid_j, LAGS_VAZAO
    )
    
    # --- C. Treinar Modelo de Volume ---
    # No treino, o modelo aprende a relação: Vol_t = f(Vol_t-1, Vn_t_real, Vj_t_real)
    model_volume = LinearRegression()
    #model_volume = MLPRegressor()
    model_volume = treinar_modelo_volume(
        df_train, nome_coluna_volume, nome_coluna_vazao_natural, nome_coluna_vazao_jusante, model_volume
    )

    # =================================================================
    # ETAPA 2: PREVISÃO RECURSIVA (Passo a passo no Teste)
    # =================================================================
    
    # Prepara as features determinísticas para TODO o período de teste (pois sabemos as datas)
    X_trend_test_n = criar_features_deterministicas(df_test.index)
    X_trend_test_j = criar_features_deterministicas(df_test.index)
    X_trend_test_v = criar_features_deterministicas_volume(df_test.index)
    
    # Inicializa históricos para a recursão
    # Precisamos do histórico de resíduos das vazões (buffer) e do último volume conhecido
    hist_resid_n = list(last_resids_n) # Ex: [resid_t-2, resid_t-1]
    hist_resid_j = list(last_resids_j)
    last_vol = df_train[nome_coluna_volume].iloc[-1] # Volume t-1 inicial
    
    preds_volume = []
    preds_vn = []
    preds_vj = []
    # Nomes das colunas de lag (para criar o DF certinho e calar o warning)
    cols_lags = [f'lag_{k}' for k in range(1, LAGS_VAZAO + 1)]
    
    # LOOP DIA A DIA (Aqui acontece a mágica recursiva)
    for i in range(len(df_test)):
        current_date = df_test.index[i]
        idx_atual = [df_test.index[i]]
        
        # --- 1. Prever Vazão Natural ---
        # Parte 1: Tendência (Determinística)
        trend_val_n = m_trend_n.predict(X_trend_test_n.iloc[[i]])[0]
        # Resíduo: CRIAR DATAFRAME COM NOMES DE COLUNAS
        vals_lags_n = np.array(hist_resid_n[-LAGS_VAZAO:][::-1])
        df_feat_resid_n = pd.DataFrame([vals_lags_n], columns=cols_lags, index=idx_atual)
        # Parte 2: Resíduo (Baseado nos lags anteriores previstos ou calculados)
        # Monta array de features [lag_1, lag_2] (invertendo a lista para ficar t-1, t-2)
        resid_pred_n = m_resid_n.predict(df_feat_resid_n)[0]
        
        # Valor Final
        y_pred_n = trend_val_n + resid_pred_n
        preds_vn.append(y_pred_n)
        
        # Atualiza histórico de resíduos (Assumimos que o resíduo "real" é o predito para continuar o loop s/ dados futuros)
        # Opcional: Se quiser atualizar com erro zero (puro autoregressivo), apenas appenda o predito.
        hist_resid_n.append(resid_pred_n) 
        
        # --- 2. Prever Vazão Jusante ---
        trend_val_j = m_trend_j.predict(X_trend_test_j.iloc[[i]])[0]
        vals_lags_j = np.array(hist_resid_j[-LAGS_VAZAO:][::-1])
        df_feat_resid_j = pd.DataFrame([vals_lags_j], columns=cols_lags, index=idx_atual)
        resid_pred_j = m_resid_j.predict(df_feat_resid_j)[0]
        
        y_pred_j = trend_val_j + resid_pred_j
        preds_vj.append(y_pred_j)
        hist_resid_j.append(resid_pred_j)
        
        # --- 3. Prever Volume ---
        # Features: [Vol_lag1, Vazao_N_pred, Vazao_J_pred]
        # Parte Dinâmica (calculada agora)
        df_dynamic = pd.DataFrame({
            'vol_lag1': [last_vol],
            'vazao_n': [y_pred_n],
            'vazao_j': [y_pred_j]
        }, index=idx_atual)
        
        # Parte Estática (já calculada antes, pegamos a linha 'i')
        # CORREÇÃO CRÍTICA: troquei iloc[[1]] por iloc[[i]]
        df_static = X_trend_test_v.iloc[[i]] 
        
        # Concatenar (Pandas alinha tudo e mantém os nomes das colunas originais)
        X_vol_input = pd.concat([df_dynamic, df_static], axis=1)
        
        vol_pred = model_volume.predict(X_vol_input)[0]
        preds_volume.append(vol_pred)
        
        # Atualiza o last_vol para o próximo passo ser o volume predito agora
        last_vol = vol_pred

    # =================================================================
    # ETAPA 3: AVALIAÇÃO
    # =================================================================
    y_true = df_test[nome_coluna_volume]
    rmse = np.sqrt(mean_squared_error(y_true, preds_volume))
    mae = np.mean(np.abs(y_true - preds_volume))
    resultados.append(rmse)
    print(f"RMSE Fold {fold+1}: {rmse:.4f}")
    print(f"MAE Fold {fold+1}: {mae:.4f}")
    print("-" * 30)
    # === PLOT DE CADA FOLD ===
    plt.figure(figsize=(12, 5))
    plt.plot(y_true.index, y_true, label='Real (Observado)', color='navy', linewidth=2)
    plt.plot(y_true.index, preds_volume, label='Previsão Recursiva', color='darkorange', linestyle='--', linewidth=2)
    
    plt.title(f'Fold {fold+1} - Horizonte: {VALIDATION_SIZE} dias | RMSE: {rmse:.2f}')
    plt.xlabel('Data')
    plt.ylabel(nome_coluna_volume)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show() # Exibe o gráfico antes de passar para o próximo fold
    
    print("-" * 30)

print(f"\nRMSE Médio: {np.mean(resultados):.4f}")

In [ ]:
### Volume atual - Volume anterior -> Variação do volume
### Variação do Volume atual - Variação do Volume anterior -> Variação da variação do volume

df[nome_coluna_volume].plot()

In [ ]:
k = df.reset_index()
k = k[(k['Data']<='2014-03-01')|(k['Data']>='2016-02-15')]
k.set_index('Data', inplace = True)
k['Volume Útil Armazenado (%)'].plot()

In [ ]:
df.reset_index()

# Usando vazao com fator de correcao.

Falta fazer gridsearch iterando varios modelos e parametros inclusive esse fator.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.metrics import mean_squared_error
import warnings
import matplotlib.pyplot as plt

# Ignore warnings that match the specified criteria
warnings.filterwarnings('ignore', message='.*deprecated.*', category=DeprecationWarning)

# Issue a warning
warnings.warn('This is a deprecated feature', DeprecationWarning)

# --- 1. CONFIGURAÇÃO E DADOS ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df[(df['Data']<='2014-03-01')|(df['Data']>='2016-02-15')]
df.set_index("Data", inplace=True)
#(k['Data']<='2014-03-01')|(k['Data']>='2016-02-15')

#df = df[df.index >= "2018-01-01"]

VALIDATION_SIZE = 90
LAGS_VAZAO = 2
LAGS_VOLUME = 1

# --- 2. FUNÇÕES AUXILIARES (DESACOPLADAS) ---

def criar_features_deterministicas(index, order=2):
    """Cria X para sazonalidade e tendência (usado pelos modelos lineares de vazão)."""
    fourier = CalendarFourier(freq="YE", order=order)
    dp = DeterministicProcess(
        index=index,
        constant=True,
        order=1, # Tendência linear
        seasonal=False,
        additional_terms=[fourier],
        drop=True,
    )
    return dp.in_sample()

def criar_features_deterministicas_volume(index):
    dp = DeterministicProcess(
        index=index,
        constant=True, order=1,
        additional_terms=[
            Fourier(period=365.25 * 2, order=1),
            CalendarFourier(freq="YE", order=2)
        ],
        drop=True
    )
    return dp.in_sample()

def criar_lags(series, lags):
    """Cria DataFrame com lags para treino."""
    df_lags = pd.DataFrame(index=series.index)
    for lag in range(1, lags + 1):
        df_lags[f'lag_{lag}'] = series.shift(lag)
    return df_lags

def treinar_modelo_hibrido_vazao(y_train, modelo_trend, modelo_resid, lags):
    """
    Treina o combo: Modelo Linear (Tendência) + Modelo ML (Resíduos Lags).
    Retorna os modelos treinados e os últimos resíduos para iniciar o loop de teste.
    """
    # 1. Features Determinísticas
    X_trend = criar_features_deterministicas(y_train.index)
    
    # 2. Fit Modelo Tendência
    modelo_trend.fit(X_trend, y_train)
    y_pred_trend = modelo_trend.predict(X_trend)
    y_pred_trend = pd.Series(y_pred_trend, index=y_train.index)
    
    # 3. Calcular Resíduos
    residuos = y_train - y_pred_trend
    
    # 4. Criar Lags dos Resíduos para o Modelo 2
    X_resid = criar_lags(residuos, lags).dropna()
    y_resid_target = residuos.loc[X_resid.index]
    
    # 5. Fit Modelo Resíduo
    modelo_resid.fit(X_resid, y_resid_target)
    
    # Retorna modelos e os ultimos dados de residuos para servir de 'semente' no teste
    last_residuals = residuos.tail(lags).values
    
    return modelo_trend, modelo_resid, last_residuals

def treinar_modelo_volume(df_train, col_vol, col_vn, col_vj, model_vol):
    y = df_train[col_vol]
    
    X_dynamic = pd.DataFrame(index=df_train.index)
    X_dynamic['vol_lag1'] = df_train[col_vol].shift(1)
    
    # --- NOVAS FEATURES ---
    # 1ª Derivada: Volume(t-1) - Volume(t-2)
    X_dynamic['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # 2ª Derivada: (Vol(t-1) - Vol(t-2)) - (Vol(t-2) - Vol(t-3))
    # Basicamente: diff(t-1) - diff(t-2)
    X_dynamic['vol_diff2'] = (df_train[col_vol].shift(1) - df_train[col_vol].shift(2)) - \
                             (df_train[col_vol].shift(2) - df_train[col_vol].shift(3))
    # ----------------------

    X_dynamic['vazao_n'] = df_train[col_vn] 
    X_dynamic['vazao_j'] = df_train[col_vj] 
    
    X_trend = criar_features_deterministicas_volume(df_train.index)
    
    X_full = pd.concat([X_dynamic, X_trend], axis=1).dropna()
    y_train = y.loc[X_full.index]
    
    model_vol.fit(X_full, y_train)
    return model_vol, X_full


def mostrar_equacao_linear(modelo, nomes_colunas):
    # 1. Pega os coeficientes e o intercepto
    coeficientes = modelo.coef_
    intercepto = modelo.intercept_
    
    # 2. Cria um DataFrame para facilitar a leitura
    df_params = pd.DataFrame({
        'Variável': nomes_colunas,
        'Peso (Coeficiente)': coeficientes
    })
    
    # 3. Ordena pelo peso absoluto (para ver o que impacta mais)
    df_params['Impacto Absoluto'] = df_params['Peso (Coeficiente)'].abs()
    df_params = df_params.sort_values(by='Impacto Absoluto', ascending=False).drop(columns='Impacto Absoluto')
    
    print("=== EQUAÇÃO DO MODELO LINEAR ===")
    print(f"Intercepto (Valor Base): {intercepto:.4f}")
    print("-" * 30)
    print(df_params)
    print("-" * 30)
    
    # 4. Monta a string da equação visualmente
    equacao_str = f"Y = {intercepto:.4f}"
    for feat, coef in zip(nomes_colunas, coeficientes):
        sinal = "+" if coef >= 0 else "-"
        equacao_str += f" {sinal} ({abs(coef):.4f} * {feat})"
    
    print("\nFórmula aproximada:")
    print(equacao_str)


# --- 3. LOOP DE VALIDAÇÃO (TIME SERIES SPLIT) ---

tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)

resultados = []

print("Iniciando Validação Cruzada...\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
    # Separação dos dados
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]
    
    print(f"Fold {fold+1}: Treino até {df_train.index[-1].date()} | Teste de {df_test.index[0].date()} a {df_test.index[-1].date()}")

    # =================================================================
    # ETAPA 1: TREINAMENTO (Usando apenas dados de Treino)
    # =================================================================
    
    # --- A. Treinar Vazão Natural (Híbrido) ---
    model_trend_n = Lasso(alpha=0.1)
    model_resid_n = KNeighborsRegressor(n_neighbors=5)
    
    m_trend_n, m_resid_n, last_resids_n = treinar_modelo_hibrido_vazao(
        df_train[nome_coluna_vazao_natural], model_trend_n, model_resid_n, LAGS_VAZAO
    )
    
    # --- B. Treinar Vazão Jusante (Híbrido) ---
    model_trend_j = LinearRegression()
    model_resid_j = RandomForestRegressor(n_estimators=50, random_state=42)
    
    m_trend_j, m_resid_j, last_resids_j = treinar_modelo_hibrido_vazao(
        df_train[nome_coluna_vazao_jusante], model_trend_j, model_resid_j, LAGS_VAZAO
    )
    
    # --- C. Treinar Modelo de Volume ---
    # No treino, o modelo aprende a relação: Vol_t = f(Vol_t-1, Vn_t_real, Vj_t_real)
    model_volume = LinearRegression()
    #model_volume = MLPRegressor()
    model_volume, X_full_eq = treinar_modelo_volume(
        df_train, nome_coluna_volume, nome_coluna_vazao_natural, nome_coluna_vazao_jusante, model_volume
    )
    mostrar_equacao_linear(model_volume, X_full_eq.columns)

    # =================================================================
    # ETAPA 2: PREVISÃO RECURSIVA (Passo a passo no Teste)
    # =================================================================
    
    # Prepara as features determinísticas para TODO o período de teste (pois sabemos as datas)
    X_trend_test_n = criar_features_deterministicas(df_test.index)
    X_trend_test_j = criar_features_deterministicas(df_test.index)
    X_trend_test_v = criar_features_deterministicas_volume(df_test.index)
    
    # Inicializa históricos para a recursão
    hist_resid_n = list(last_resids_n)
    hist_resid_j = list(last_resids_j)
    
    # --- MUDANÇA AQUI: BUFFER DE VOLUME ---
    # Precisamos dos ultimos 3 dias para calcular a 2ª derivada inicial
    # [t-3, t-2, t-1]
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    
    preds_volume = []
    preds_vn = []
    preds_vj = []
    # Nomes das colunas de lag (para criar o DF certinho e calar o warning)
    cols_lags = [f'lag_{k}' for k in range(1, LAGS_VAZAO + 1)]
    
    # LOOP DIA A DIA (Aqui acontece a mágica recursiva)
    for i in range(len(df_test)):
        current_date = df_test.index[i]
        idx_atual = [df_test.index[i]]
        
        # --- 1. Prever Vazão Natural ---
        # Parte 1: Tendência (Determinística)
        trend_val_n = m_trend_n.predict(X_trend_test_n.iloc[[i]])[0]
        # Resíduo: CRIAR DATAFRAME COM NOMES DE COLUNAS
        vals_lags_n = np.array(hist_resid_n[-LAGS_VAZAO:][::-1])
        df_feat_resid_n = pd.DataFrame([vals_lags_n], columns=cols_lags, index=idx_atual)
        # Parte 2: Resíduo (Baseado nos lags anteriores previstos ou calculados)
        # Monta array de features [lag_1, lag_2] (invertendo a lista para ficar t-1, t-2)
        resid_pred_n = m_resid_n.predict(df_feat_resid_n)[0]
        
        # Valor Final
        y_pred_n = trend_val_n + resid_pred_n
        preds_vn.append(y_pred_n)
        
        # Atualiza histórico de resíduos (Assumimos que o resíduo "real" é o predito para continuar o loop s/ dados futuros)
        # Opcional: Se quiser atualizar com erro zero (puro autoregressivo), apenas appenda o predito.
        hist_resid_n.append(resid_pred_n) 
        
        # --- 2. Prever Vazão Jusante ---
        trend_val_j = m_trend_j.predict(X_trend_test_j.iloc[[i]])[0]
        vals_lags_j = np.array(hist_resid_j[-LAGS_VAZAO:][::-1])
        df_feat_resid_j = pd.DataFrame([vals_lags_j], columns=cols_lags, index=idx_atual)
        resid_pred_j = m_resid_j.predict(df_feat_resid_j)[0]
        
        y_pred_j = trend_val_j + resid_pred_j
        preds_vj.append(y_pred_j)
        hist_resid_j.append(resid_pred_j)
        
        # --- 3. Prever Volume ---
        # Features: [Vol_lag1, Vazao_N_pred, Vazao_J_pred]
        # Parte Dinâmica (calculada agora)
        # Recupera valores do buffer
        vol_t_1 = buffer_vols[-1] # Ontem
        vol_t_2 = buffer_vols[-2] # Anteontem
        vol_t_3 = buffer_vols[-3] # Ante-anteontem
        
        # Calcula as derivadas
        feat_diff = vol_t_1 - vol_t_2
        feat_diff_prev = vol_t_2 - vol_t_3
        feat_diff2 = feat_diff - feat_diff_prev
        
        df_dynamic = pd.DataFrame({
            'vol_lag1':  [vol_t_1],
            'vol_diff':  [feat_diff],  # Nova feature
            'vol_diff2': [feat_diff2], # Nova feature
            'vazao_n':   [y_pred_n],
            'vazao_j':   [y_pred_j]
        }, index=idx_atual)
        
        # Parte Estática (já calculada antes, pegamos a linha 'i')
        # CORREÇÃO CRÍTICA: troquei iloc[[1]] por iloc[[i]]
        df_static = X_trend_test_v.iloc[[i]] 
        
        # Concatenar (Pandas alinha tudo e mantém os nomes das colunas originais)
        X_vol_input = pd.concat([df_dynamic, df_static], axis=1)

        
        vol_pred = model_volume.predict(X_vol_input)[0]
        preds_volume.append(vol_pred)
        
        # Atualiza o last_vol para o próximo passo ser o volume predito agora
        # --- ATUALIZA O BUFFER ---
        buffer_vols.append(vol_pred) # Adiciona o novo previsto no fim
        buffer_vols.pop(0)           # Remove o mais antigo (desliza a janela)

    # =================================================================
    # ETAPA 3: AVALIAÇÃO
    # =================================================================
    y_true = df_test[nome_coluna_volume]
    rmse = np.sqrt(mean_squared_error(y_true, preds_volume))
    mae = np.mean(np.abs(y_true - preds_volume))
    resultados.append(rmse)
    print(f"RMSE Fold {fold+1}: {rmse:.4f}")
    print(f"MAE Fold {fold+1}: {mae:.4f}")
    print(model_volume.coef_)
    print("-" * 30)
    
    # === PLOT DE CADA FOLD ===
    plt.figure(figsize=(12, 5))
    plt.plot(y_true.index, y_true, label='Real (Observado)', color='navy', linewidth=2)
    plt.plot(y_true.index, preds_volume, label='Previsão Recursiva', color='darkorange', linestyle='--', linewidth=2)
    
    plt.title(f'Fold {fold+1} - Horizonte: {VALIDATION_SIZE} dias | RMSE: {rmse:.2f}')
    plt.xlabel('Data')
    plt.ylabel(nome_coluna_volume)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show() # Exibe o gráfico antes de passar para o próximo fold
    
    print("-" * 30)

print(f"\nRMSE Médio: {np.mean(resultados):.4f}")

### AVALIANDO MODELO VAZAO

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.metrics import mean_squared_error
import warnings
import matplotlib.pyplot as plt

# Ignore warnings that match the specified criteria
warnings.filterwarnings('ignore', message='.*deprecated.*', category=DeprecationWarning)

# Issue a warning
warnings.warn('This is a deprecated feature', DeprecationWarning)

# --- 1. CONFIGURAÇÃO E DADOS ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df[(df['Data']<='2014-03-01')|(df['Data']>='2016-02-15')]
df.set_index("Data", inplace=True)
#(k['Data']<='2014-03-01')|(k['Data']>='2016-02-15')

#df = df[df.index >= "2018-01-01"]

VALIDATION_SIZE = 90
LAGS_VAZAO = 2
LAGS_VOLUME = 1

# --- 2. FUNÇÕES AUXILIARES (DESACOPLADAS) ---

def criar_features_deterministicas(index, order=2):
    """Cria X para sazonalidade e tendência (usado pelos modelos lineares de vazão)."""
    fourier = CalendarFourier(freq="YE", order=order)
    dp = DeterministicProcess(
        index=index,
        constant=True,
        order=1, # Tendência linear
        seasonal=False,
        additional_terms=[fourier],
        drop=True,
    )
    return dp.in_sample()

def criar_features_deterministicas_volume(index):
    dp = DeterministicProcess(
        index=index,
        constant=True, order=1,
        additional_terms=[
            Fourier(period=365.25 * 2, order=1),
            CalendarFourier(freq="YE", order=2)
        ],
        drop=True
    )
    return dp.in_sample()

def criar_lags(series, lags):
    """Cria DataFrame com lags para treino."""
    df_lags = pd.DataFrame(index=series.index)
    for lag in range(1, lags + 1):
        df_lags[f'lag_{lag}'] = series.shift(lag)
    return df_lags

def treinar_modelo_hibrido_vazao(y_train, modelo_trend, modelo_resid, lags):
    """
    Treina o combo: Modelo Linear (Tendência) + Modelo ML (Resíduos Lags).
    Retorna os modelos treinados e os últimos resíduos para iniciar o loop de teste.
    """
    # 1. Features Determinísticas
    X_trend = criar_features_deterministicas(y_train.index)
    
    # 2. Fit Modelo Tendência
    modelo_trend.fit(X_trend, y_train)
    y_pred_trend = modelo_trend.predict(X_trend)
    y_pred_trend = pd.Series(y_pred_trend, index=y_train.index)
    
    # 3. Calcular Resíduos
    residuos = y_train - y_pred_trend
    
    # 4. Criar Lags dos Resíduos para o Modelo 2
    X_resid = criar_lags(residuos, lags).dropna()
    y_resid_target = residuos.loc[X_resid.index]
    
    # 5. Fit Modelo Resíduo
    modelo_resid.fit(X_resid, y_resid_target)
    
    # Retorna modelos e os ultimos dados de residuos para servir de 'semente' no teste
    last_residuals = residuos.tail(lags).values
    
    return modelo_trend, modelo_resid, last_residuals

def treinar_modelo_volume(df_train, col_vol, col_vn, col_vj, model_vol):
    y = df_train[col_vol]
    
    X_dynamic = pd.DataFrame(index=df_train.index)
    X_dynamic['vol_lag1'] = df_train[col_vol].shift(1)
    
    # --- NOVAS FEATURES ---
    # 1ª Derivada: Volume(t-1) - Volume(t-2)
    X_dynamic['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # 2ª Derivada: (Vol(t-1) - Vol(t-2)) - (Vol(t-2) - Vol(t-3))
    # Basicamente: diff(t-1) - diff(t-2)
    X_dynamic['vol_diff2'] = (df_train[col_vol].shift(1) - df_train[col_vol].shift(2)) - \
                             (df_train[col_vol].shift(2) - df_train[col_vol].shift(3))
    # ----------------------

    X_dynamic['vazao_n'] = df_train[col_vn] 
    #X_dynamic['vazao_j'] = df_train[col_vj] 
    
    X_trend = criar_features_deterministicas_volume(df_train.index)
    
    X_full = pd.concat([X_dynamic, X_trend], axis=1).dropna()
    y_train = y.loc[X_full.index]
    
    model_vol.fit(X_full, y_train)
    return model_vol, X_full


def mostrar_equacao_linear(modelo, nomes_colunas):
    # 1. Pega os coeficientes e o intercepto
    coeficientes = modelo.coef_
    intercepto = modelo.intercept_
    
    # 2. Cria um DataFrame para facilitar a leitura
    df_params = pd.DataFrame({
        'Variável': nomes_colunas,
        'Peso (Coeficiente)': coeficientes
    })
    
    # 3. Ordena pelo peso absoluto (para ver o que impacta mais)
    df_params['Impacto Absoluto'] = df_params['Peso (Coeficiente)'].abs()
    df_params = df_params.sort_values(by='Impacto Absoluto', ascending=False).drop(columns='Impacto Absoluto')
    
    print("=== EQUAÇÃO DO MODELO LINEAR ===")
    print(f"Intercepto (Valor Base): {intercepto:.4f}")
    print("-" * 30)
    print(df_params)
    print("-" * 30)
    
    # 4. Monta a string da equação visualmente
    equacao_str = f"Y = {intercepto:.4f}"
    for feat, coef in zip(nomes_colunas, coeficientes):
        sinal = "+" if coef >= 0 else "-"
        equacao_str += f" {sinal} ({abs(coef):.4f} * {feat})"
    
    print("\nFórmula aproximada:")
    print(equacao_str)


# --- 3. LOOP DE VALIDAÇÃO (TIME SERIES SPLIT) ---

tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)

resultados = []

print("Iniciando Validação Cruzada...\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
    # Separação dos dados
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]
    
    print(f"Fold {fold+1}: Treino até {df_train.index[-1].date()} | Teste de {df_test.index[0].date()} a {df_test.index[-1].date()}")

    # =================================================================
    # ETAPA 1: TREINAMENTO (Usando apenas dados de Treino)
    # =================================================================
    
    # --- A. Treinar Vazão Natural (Híbrido) ---
    model_trend_n = Lasso(alpha=0.1)
    model_resid_n = KNeighborsRegressor(n_neighbors=5)
    
    m_trend_n, m_resid_n, last_resids_n = treinar_modelo_hibrido_vazao(
        df_train[nome_coluna_vazao_natural], model_trend_n, model_resid_n, LAGS_VAZAO
    )
    
    # --- B. Treinar Vazão Jusante (Híbrido) ---
   # model_trend_j = LinearRegression()
   # model_resid_j = RandomForestRegressor(n_estimators=50, random_state=42)
    
   # m_trend_j, m_resid_j, last_resids_j = treinar_modelo_hibrido_vazao(
   #     df_train[nome_coluna_vazao_jusante], model_trend_j, model_resid_j, LAGS_VAZAO
   # )
    
    # --- C. Treinar Modelo de Volume ---
    # No treino, o modelo aprende a relação: Vol_t = f(Vol_t-1, Vn_t_real, Vj_t_real)
    model_volume = LinearRegression()
    #model_volume = MLPRegressor()
    model_volume, X_full_eq = treinar_modelo_volume(
        df_train, nome_coluna_volume, nome_coluna_vazao_natural, nome_coluna_vazao_jusante, model_volume
    )
    mostrar_equacao_linear(model_volume, X_full_eq.columns)

    # =================================================================
    # ETAPA 2: PREVISÃO RECURSIVA (Passo a passo no Teste)
    # =================================================================
    
    # Prepara as features determinísticas para TODO o período de teste (pois sabemos as datas)
    X_trend_test_n = criar_features_deterministicas(df_test.index)
    X_trend_test_j = criar_features_deterministicas(df_test.index)
    X_trend_test_v = criar_features_deterministicas_volume(df_test.index)
    
    # Inicializa históricos para a recursão
    hist_resid_n = list(last_resids_n)
    hist_resid_j = list(last_resids_j)
    
    # --- MUDANÇA AQUI: BUFFER DE VOLUME ---
    # Precisamos dos ultimos 3 dias para calcular a 2ª derivada inicial
    # [t-3, t-2, t-1]
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    
    preds_volume = []
    preds_vn = []
    preds_vj = []
    # Nomes das colunas de lag (para criar o DF certinho e calar o warning)
    cols_lags = [f'lag_{k}' for k in range(1, LAGS_VAZAO + 1)]
    
    # LOOP DIA A DIA (Aqui acontece a mágica recursiva)
    for i in range(len(df_test)):
        current_date = df_test.index[i]
        idx_atual = [df_test.index[i]]
        
        # --- 1. Prever Vazão Natural ---
        # Parte 1: Tendência (Determinística)
        trend_val_n = m_trend_n.predict(X_trend_test_n.iloc[[i]])[0]
        # Resíduo: CRIAR DATAFRAME COM NOMES DE COLUNAS
        vals_lags_n = np.array(hist_resid_n[-LAGS_VAZAO:][::-1])
        df_feat_resid_n = pd.DataFrame([vals_lags_n], columns=cols_lags, index=idx_atual)
        # Parte 2: Resíduo (Baseado nos lags anteriores previstos ou calculados)
        # Monta array de features [lag_1, lag_2] (invertendo a lista para ficar t-1, t-2)
        resid_pred_n = m_resid_n.predict(df_feat_resid_n)[0]
        
        # Valor Final
        y_pred_n = trend_val_n + resid_pred_n
        preds_vn.append(y_pred_n)
        
        # Atualiza histórico de resíduos (Assumimos que o resíduo "real" é o predito para continuar o loop s/ dados futuros)
        # Opcional: Se quiser atualizar com erro zero (puro autoregressivo), apenas appenda o predito.
        hist_resid_n.append(resid_pred_n) 
        
        # --- 2. Prever Vazão Jusante ---
        trend_val_j = m_trend_j.predict(X_trend_test_j.iloc[[i]])[0]
        vals_lags_j = np.array(hist_resid_j[-LAGS_VAZAO:][::-1])
        df_feat_resid_j = pd.DataFrame([vals_lags_j], columns=cols_lags, index=idx_atual)
        resid_pred_j = m_resid_j.predict(df_feat_resid_j)[0]
        
        y_pred_j = trend_val_j + resid_pred_j
        preds_vj.append(y_pred_j)
        hist_resid_j.append(resid_pred_j)
        
        # --- 3. Prever Volume ---
        # Features: [Vol_lag1, Vazao_N_pred, Vazao_J_pred]
        # Parte Dinâmica (calculada agora)
        # Recupera valores do buffer
        vol_t_1 = buffer_vols[-1] # Ontem
        vol_t_2 = buffer_vols[-2] # Anteontem
        vol_t_3 = buffer_vols[-3] # Ante-anteontem
        
        # Calcula as derivadas
        feat_diff = vol_t_1 - vol_t_2
        feat_diff_prev = vol_t_2 - vol_t_3
        feat_diff2 = feat_diff - feat_diff_prev
        
        df_dynamic = pd.DataFrame({
            'vol_lag1':  [vol_t_1],
            'vol_diff':  [feat_diff],  # Nova feature
            'vol_diff2': [feat_diff2], # Nova feature
            'vazao_n':   [y_pred_n],
       #     'vazao_j':   [y_pred_j]
        }, index=idx_atual)
        
        # Parte Estática (já calculada antes, pegamos a linha 'i')
        # CORREÇÃO CRÍTICA: troquei iloc[[1]] por iloc[[i]]
        df_static = X_trend_test_v.iloc[[i]] 
        
        # Concatenar (Pandas alinha tudo e mantém os nomes das colunas originais)
        X_vol_input = pd.concat([df_dynamic, df_static], axis=1)

        
        vol_pred = model_volume.predict(X_vol_input)[0]
        preds_volume.append(vol_pred)
        
        # Atualiza o last_vol para o próximo passo ser o volume predito agora
        # --- ATUALIZA O BUFFER ---
        buffer_vols.append(vol_pred) # Adiciona o novo previsto no fim
        buffer_vols.pop(0)           # Remove o mais antigo (desliza a janela)


# =================================================================
    # ETAPA 3: AVALIAÇÃO DETALHADA (Volume + Vazões)
    # =================================================================
    
    # --- 1. Avaliação do VOLUME ---
    y_true_vol = df_test[nome_coluna_volume]
    rmse_vol = np.sqrt(mean_squared_error(y_true_vol, preds_volume))
    mae_vol = np.mean(np.abs(y_true_vol - preds_volume))
    resultados.append(rmse_vol)
    
    # --- 2. Avaliação da VAZÃO NATURAL ---
    y_true_vn = df_test[nome_coluna_vazao_natural]
    rmse_vn = np.sqrt(mean_squared_error(y_true_vn, preds_vn))
    mae_vn = np.mean(np.abs(y_true_vn - preds_vn))
    
    # --- 3. Avaliação da VAZÃO JUSANTE ---
    y_true_vj = df_test[nome_coluna_vazao_jusante]
    rmse_vj = np.sqrt(mean_squared_error(y_true_vj, preds_vj))
    mae_vj = np.mean(np.abs(y_true_vj - preds_vj))

    # --- PRINTS DE MÉTRICAS ---
    print(f"--- RESULTADOS FOLD {fold+1} ---")
    print(f"[VOLUME] RMSE: {rmse_vol:.4f} | MAE: {mae_vol:.4f}")
    print(f"[NATURAL] RMSE: {rmse_vn:.4f} | MAE: {mae_vn:.4f}")
    print(f"[JUSANTE] RMSE: {rmse_vj:.4f} | MAE: {mae_vj:.4f}")
    
    # Mostra os pesos do modelo linear de volume (para ver se Vazão tem peso alto)
    print("\nPesos do Modelo de Volume:")
    print(pd.Series(model_volume.coef_, index=X_full_eq.columns).sort_values(ascending=False))
    print("-" * 30)
    
    # =================================================================
    # PLOTAGEM TRIPLA (Diagnóstico de Causa Raiz)
    # =================================================================
    fig, axs = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
    
    # Gráfico 1: Volume (Onde você vê o erro final)
    axs[0].plot(y_true_vol.index, y_true_vol, label='Real', color='navy', linewidth=2)
    axs[0].plot(y_true_vol.index, preds_volume, label='Previsão Recursiva', color='darkorange', linestyle='--', linewidth=2)
    axs[0].set_title(f'Fold {fold+1} - VOLUME (RMSE: {rmse_vol:.2f})')
    axs[0].set_ylabel('% Volume')
    axs[0].grid(True, alpha=0.3)
    axs[0].legend()

    # Gráfico 2: Vazão Natural (A fonte de entrada de água)
    axs[1].plot(y_true_vn.index, y_true_vn, label='Real', color='green', alpha=0.7)
    axs[1].plot(y_true_vn.index, preds_vn, label='Previsto', color='red', linestyle='--', alpha=0.8)
    axs[1].set_title(f'Diagnóstico: VAZÃO NATURAL (RMSE: {rmse_vn:.2f})')
    axs[1].set_ylabel('m³/s')
    axs[1].grid(True, alpha=0.3)
    axs[1].legend()

    # Gráfico 3: Vazão Jusante (A saída de água)
    #axs[2].plot(y_true_vj.index, y_true_vj, label='Real', color='purple', alpha=0.7)
    #axs[2].plot(y_true_vj.index, preds_vj, label='Previsto', color='magenta', linestyle='--', alpha=0.8)
    #axs[2].set_title(f'Diagnóstico: VAZÃO JUSANTE (RMSE: {rmse_vj:.2f})')
    #axs[2].set_ylabel('m³/s')
    #axs[2].set_xlabel('Data')
    #axs[2].grid(True, alpha=0.3)
    #axs[2].legend()

    plt.tight_layout()
    plt.show() # Exibe o gráfico antes de passar para o próximo fold
    
    print("=" * 50)

print(f"\nRMSE Médio (Volume): {np.mean(resultados):.4f}")

print(f"\nRMSE Médio: {np.mean(resultados):.4f}")

## Olhar modelo de vazao



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

# --- 1. CONFIGURAÇÃO (Ajuste conforme seu ambiente) ---
# df = ... (Seu código de carga de dados aqui)
# df.set_index("Data", inplace=True)
# Certifique-se que o índice é Datetime

col_target = "Vazão Natural (m³/s)" 
# col_target = nome_coluna_vazao_natural (usando sua variável)

# Hiperparâmetros para teste rápido
VALIDATION_SIZE = 90  # Dias para prever em cada fold
LAGS = 2              # Lags de resíduo

# --- 2. FUNÇÕES DO MODELO (Recortadas do seu código original) ---

def criar_features_deterministicas(index, order=2):
    fourier = CalendarFourier(freq="YE", order=order)
    dp = DeterministicProcess(
        index=index,
        constant=True,
        order=1, # Tente mudar para 0 se a tendência linear for ruim
        seasonal=False,
        additional_terms=[fourier],
        drop=True,
    )
    return dp.in_sample()

def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for lag in range(1, lags + 1):
        df_lags[f'lag_{lag}'] = series.shift(lag)
    return df_lags

def treinar_modelo_vazao(y_train, modelo_trend, modelo_resid, lags):
    # 1. Trend
    X_trend = criar_features_deterministicas(y_train.index)
    modelo_trend.fit(X_trend, y_train)
    y_pred_trend = modelo_trend.predict(X_trend)
    y_pred_trend = pd.Series(y_pred_trend, index=y_train.index)
    
    # 2. Resíduos
    residuos = y_train - y_pred_trend
    X_resid = criar_lags(residuos, lags).dropna()
    y_resid_target = residuos.loc[X_resid.index]
    
    modelo_resid.fit(X_resid, y_resid_target)
    
    # Retorna last residuals para o loop
    last_residuals = residuos.tail(lags).values
    return modelo_trend, modelo_resid, last_residuals

# --- 3. LOOP DE VALIDAÇÃO DEDICADO ---

tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados_metricas = []

print(f"=== INICIANDO DIAGNÓSTICO DE {col_target} ===")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]
    y_train = df_train[col_target]
    y_test = df_test[col_target]
    
    # --- A. DEFINIÇÃO DOS MODELOS (AQUI VOCÊ MEXE PARA TESTAR) ---
    # Sugestão: Tente Ridge em vez de Lasso, ou reduza o KNN
    model_trend = Lasso(alpha=0.1) 
    # model_trend = LinearRegression() # Teste sem regularização
    
    model_resid = RandomForestRegressor()
    # model_resid = RandomForestRegressor(max_depth=5, n_estimators=50) # Tente RF
    
    # --- B. TREINO ---
    m_trend, m_resid, last_resids = treinar_modelo_vazao(
        y_train, model_trend, model_resid, LAGS
    )
    
    # --- C. PREVISÃO RECURSIVA (Loop dia-a-dia) ---
    X_trend_test = criar_features_deterministicas(df_test.index)
    hist_resid = list(last_resids)
    preds = []
    
    # Cria inputs para o modelo NAIVE (Persistência: amanhã = hoje)
    # Pega o último valor do treino para começar
    last_val_known = y_train.iloc[-1]

    cols_lags = [f'lag_{k}' for k in range(1, LAGS + 1)]

    for i in range(len(df_test)):
        # 1. Componente de Tendência
        trend_val = m_trend.predict(X_trend_test.iloc[[i]])[0]
        
        # 2. Componente de Resíduo (usando lags previstos anteriormente)
        vals_lags = np.array(hist_resid[-LAGS:][::-1])
        df_feat_resid = pd.DataFrame([vals_lags], columns=cols_lags)
        resid_pred = m_resid.predict(df_feat_resid)[0]
        
        # 3. Soma
        pred_final = trend_val + resid_pred
        # TRAVA DE SEGURANÇA: Vazão não pode ser negativa
        if pred_final < 0: pred_final = 0 
        
        preds.append(pred_final)
        
        # Atualiza histórico de resíduos com o PREVISTO (recursão pura)
        hist_resid.append(resid_pred)
        

    
    # Para o gráfico ficar justo, o Naive vou plotar apenas visualmente deslocado depois.
    
    # --- D. MÉTRICAS ---
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    # CÁLCULO DE VIÉS (BIAS): Média (Previsto - Real)
    # Se Positivo: Modelo prevê pra cima. Se Negativo: Prevê pra baixo.
    bias = np.mean(np.array(preds) - y_test.values)
    
    print(f"FOLD {fold+1}: RMSE={rmse:.2f} | MAE={mae:.2f} | VIÉS={bias:.2f}")
    
    # --- E. VISUALIZAÇÃO DIAGNÓSTICA ---
    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # Plot 1: Série Temporal
    ax[0].plot(y_test.index, y_test, label='Real', color='black', alpha=0.6)
    ax[0].plot(y_test.index, preds, label='Seu Modelo', color='red', linewidth=2)
    ax[0].set_title(f"Fold {fold+1} - Real vs Previsto (Viés: {bias:.2f})")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Plot 2: Resíduos (Erro) ao longo do tempo
    # Isso mostra se o erro é aleatório ou se tem padrão (ex: erro cresce com o tempo)
    errors = np.array(preds) - y_test.values
    ax[1].plot(y_test.index, errors, color='blue', marker='o', markersize=3, linestyle='None')
    ax[1].axhline(0, color='black', linestyle='--')
    ax[1].set_title("Resíduos (Previsto - Real) por dia")
    ax[1].set_ylabel("Erro (m³/s)")
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
VALIDATION_SIZE = 90
LAGS_VAZAO = 2  # Lags para o modelo ML de vazão
LAGS_VOLUME = 1

# Supondo df já carregado
# df = ... 

# --- 1. FUNÇÕES: CLIMATOLOGIA (Feature 1) ---
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        # Filtro simples de dias
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            # Remove outliers (10% - 90%)
            vals = vals[(vals >= vals.quantile(0.10)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature(index, curva_clim):
    """Retorna array com valores da climatologia para as datas do index"""
    days = index.dayofyear
    # Ajuste para bissexto > 366 (se houver erro, usa o ultimo valor)
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

# --- 2. FUNÇÕES: MODELO ML DE VAZÃO (Feature 2) ---
def criar_feat_fourier(index):
    dp = DeterministicProcess(index=index, constant=True, order=1, 
                              additional_terms=[CalendarFourier("YE", 2)], drop=True)
    return dp.in_sample()

def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for l in range(1, lags+1):
        df_lags[f'lag_{l}'] = series.shift(l)
    return df_lags

def treinar_modelo_vazao_ml(df_train, col_target):
    # 1. Tendência
    y = df_train[col_target]
    X_trend = criar_feat_fourier(df_train.index)
    
    # Modelo de Tendência (Lasso para ser mais robusto)
    m_trend = Lasso(alpha=0.5)
    m_trend.fit(X_trend, y)
    trend_pred = pd.Series(m_trend.predict(X_trend), index=y.index)
    
    # 2. Resíduos
    resid = y - trend_pred
    X_resid = criar_lags(resid, LAGS_VAZAO).dropna()
    y_resid = resid.loc[X_resid.index]
    
    # Modelo de Resíduos (KNN ou RF)
    m_resid = KNeighborsRegressor(n_neighbors=10) # Aumentei vizinhos para suavizar
    m_resid.fit(X_resid, y_resid)
    
    last_resids = resid.tail(LAGS_VAZAO).values
    return m_trend, m_resid, last_resids

# --- 3. FUNÇÃO: MODELO DE VOLUME (O "JUIZ") ---
def treinar_modelo_volume_final(df_train, col_vol, col_vn, curva_clim, m_trend_vn, m_resid_vn):
    """
    Treina o volume usando:
    1. Lags do Volume
    2. Feature Vazão Climatologia
    3. Feature Vazão ML (In-Sample Prediction)
    """
    y = df_train[col_vol]
    
    # --- Engenharia de Features Dinâmicas ---
    X = pd.DataFrame(index=df_train.index)
    X['vol_lag1'] = df_train[col_vol].shift(1)
    X['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # --- Feature Exógena 1: Climatologia ---
    X['feat_vazao_clim'] = get_clim_feature(df_train.index, curva_clim)
    
    # --- Feature Exógena 2: Modelo ML (Predição In-Sample) ---
    # Recriamos a previsão que o modelo ML faria no treino para o Volume aprender a confiar (ou não) nele
    X_trend = criar_feat_fourier(df_train.index)
    trend_in = m_trend_vn.predict(X_trend)
    
    # Resíduos in-sample (aproximado via lags reais para treinar rápido)
    resid_real = df_train[col_vn] - trend_in
    X_resid_lags = criar_lags(resid_real, LAGS_VAZAO) # Lags reais
    # Para simplificar e evitar vazamento total, vamos usar a Vazão Real como proxy do "Melhor ML Possível"
    # ou usar a predição real. Vamos usar a predição in-sample do modelo:
    resid_pred_in = m_resid_vn.predict(X_resid_lags.fillna(0)) 
    
    X['feat_vazao_ml'] = trend_in + resid_pred_in
    
    # Drop NaNs gerados pelos lags
    X_full = X.dropna()
    y_train = y.loc[X_full.index]
    
    model_vol = LinearRegression()
    model_vol.fit(X_full, y_train)
    
    return model_vol, X_full.columns

# --- 4. LOOP DE VALIDAÇÃO ---
tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados = []

print("=== INICIANDO MODELO HÍBRIDO DUPLO (CLIM + ML) ===\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    
    # A. Treinar Climatologia
    curva_clim = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural)
    
    # B. Treinar Modelo Vazão ML
    m_vn_trend, m_vn_resid, last_resids_vn = treinar_modelo_vazao_ml(df_train, nome_coluna_vazao_natural)
    
    # C. Treinar Modelo Volume (Com as 2 features)
    model_vol, col_names = treinar_modelo_volume_final(
        df_train, nome_coluna_volume, nome_coluna_vazao_natural, 
        curva_clim, m_vn_trend, m_vn_resid
    )
    
    # Mostrar pesos no primeiro fold
    if fold == 0:
        coefs = pd.Series(model_vol.coef_, index=col_names)
        print("--- PESOS QUE O MODELO DE VOLUME DEU ---")
        print(coefs.sort_values(ascending=False))
        print("Note: Se 'feat_vazao_clim' for maior que 'feat_vazao_ml', o modelo confia mais na média histórica.\n")

    # --- PREVISÃO RECURSIVA ---
    # Preparação
    X_trend_test_vn = criar_feat_fourier(df_test.index)
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    hist_resid_vn = list(last_resids_vn)
    
    preds_vol = []
    
    # Listas para guardar as features usadas (para plotar depois)
    log_feat_clim = []
    log_feat_ml = []
    
    for i in range(len(df_test)):
        date = df_test.index[i]
        
        # 1. Feature Climatologia
        val_clim = get_clim_feature(pd.DatetimeIndex([date]), curva_clim)[0]
        
        # 2. Feature ML Recursivo
        # a. Trend
        val_trend = m_vn_trend.predict(X_trend_test_vn.iloc[[i]])[0]
        # b. Resid
        lags_resid = np.array(hist_resid_vn[-LAGS_VAZAO:][::-1])
        val_resid = m_vn_resid.predict(pd.DataFrame([lags_resid], columns=[f'lag_{k}' for k in range(1,LAGS_VAZAO+1)]))[0]
        
        val_ml = val_trend + val_resid
        if val_ml < 0: val_ml = 0 # Trava física
        
        # Atualiza histórico de resíduo do ML
        hist_resid_vn.append(val_resid)
        
        # Logs para plot
        log_feat_clim.append(val_clim)
        log_feat_ml.append(val_ml)
        
        # 3. Prever Volume
        vol_t1 = buffer_vols[-1]
        vol_diff = vol_t1 - buffer_vols[-2]
        
        # Monta input para o modelo de volume (MESMA ORDEM DO TREINO)
        # Colunas: ['vol_lag1', 'vol_diff', 'feat_vazao_clim', 'feat_vazao_ml']
        input_vol = pd.DataFrame({
            'vol_lag1': [vol_t1],
            'vol_diff': [vol_diff],
            'feat_vazao_clim': [val_clim],
            'feat_vazao_ml': [val_ml]
        })
        
        pred_vol = model_vol.predict(input_vol)[0]
        preds_vol.append(pred_vol)
        
        # Atualiza buffer volume
        buffer_vols.append(pred_vol)
        buffer_vols.pop(0)

    # --- AVALIAÇÃO ---
    rmse = np.sqrt(mean_squared_error(df_test[nome_coluna_volume], preds_vol))
    resultados.append(rmse)
    
    print(f"Fold {fold+1} RMSE: {rmse:.4f}")
    
    # Plot Diagnóstico
    fig, ax = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Gráfico 1: Volume
    ax[0].plot(df_test.index, df_test[nome_coluna_volume], label='Real', color='navy')
    ax[0].plot(df_test.index, preds_vol, label='Previsto Híbrido', color='orange', linestyle='--')
    ax[0].set_title(f"Volume - Fold {fold+1}")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Gráfico 2: Batalha das Features de Vazão
    ax[1].plot(df_test.index, df_test[nome_coluna_vazao_natural], label='Vazão REAL (Alvo)', color='black', alpha=0.3, linewidth=3)
    ax[1].plot(df_test.index, log_feat_clim, label='Feature: Climatologia', color='green')
    ax[1].plot(df_test.index, log_feat_ml, label='Feature: Modelo ML', color='red', linestyle=':')
    ax[1].set_title("Inputs Exógenos: Climatologia vs ML")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio Final: {np.mean(resultados):.4f}")

# USANDO SO CLIMATOLOGIA DA VAZAO

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.metrics import mean_squared_error
import warnings
import matplotlib.pyplot as plt

# Ignore warnings
warnings.filterwarnings('ignore', message='.*deprecated.*', category=DeprecationWarning)

# --- 1. CONFIGURAÇÃO E DADOS ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

# Simulação de carga (Substitua pelo seu import)
# df = import_dataframe() 
# df[(df['Data']<='2014-03-01')|(df['Data']>='2016-02-15')]
# df.set_index("Data", inplace=True)

# Supondo que df já esteja carregado no seu ambiente:
# Certifique-se que o índice é datetime

VALIDATION_SIZE = 90
LAGS_VOLUME = 1

# --- 2. FUNÇÕES AUXILIARES ---

def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    """
    Calcula a média histórica para cada dia do ano (1-366),
    usando uma janela deslizante e removendo outliers (quantis).
    """
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    
    climatologia = {}
    
    for day in range(1, 367):
        # Janela deslizante para capturar dias vizinhos (suavização)
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        
        # Filtra (considerando dias do ano)
        # Nota: simplificado para não quebrar na virada do ano, mas funcional
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        valores = temp.loc[mask, col_target]
        
        if len(valores) > 0:
            # Remove outliers extremos (anos de seca ou cheia recorde que distorcem a média)
            q_low = valores.quantile(0.10)
            q_high = valores.quantile(0.90)
            valores_filtrados = valores[(valores >= q_low) & (valores <= q_high)]
            
            if len(valores_filtrados) == 0:
                media = valores.mean()
            else:
                media = valores_filtrados.mean()
        else:
            media = 0
            
        climatologia[day] = media
        
    return pd.Series(climatologia, name='climatologia')

def criar_features_deterministicas_volume(index):
    # Usado apenas para features de sazonalidade dentro do modelo de volume
    dp = DeterministicProcess(
        index=index,
        constant=True, order=1,
        additional_terms=[
            # Sazonalidade anual e bienal para ajudar o volume
            Fourier(period=365.25 * 2, order=1),
            CalendarFourier(freq="YE", order=2)
        ],
        drop=True
    )
    return dp.in_sample()

def treinar_modelo_volume(df_train, col_vol, col_vn, model_vol):
    """
    Treina o modelo de Volume.
    IMPORTANTE: Treinamos com a Vazão REAL histórica.
    O modelo aprende a física: "Se entrar X água, o volume sobe Y".
    No teste, nós mentimos para ele entregando a Climatologia no lugar da Vazão Real.
    """
    y = df_train[col_vol]
    
    X_dynamic = pd.DataFrame(index=df_train.index)
    X_dynamic['vol_lag1'] = df_train[col_vol].shift(1)
    
    # 1ª Derivada
    X_dynamic['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    # 2ª Derivada
    X_dynamic['vol_diff2'] = (df_train[col_vol].shift(1) - df_train[col_vol].shift(2)) - \
                             (df_train[col_vol].shift(2) - df_train[col_vol].shift(3))
    
    # Input Exógeno (Vazão Natural)
    X_dynamic['vazao_n'] = df_train[col_vn] 
    
    # Features Determinísticas (Sazonalidade do Volume)
    X_trend = criar_features_deterministicas_volume(df_train.index)
    
    X_full = pd.concat([X_dynamic, X_trend], axis=1).dropna()
    y_train = y.loc[X_full.index]
    
    model_vol.fit(X_full, y_train)
    return model_vol, X_full

def mostrar_equacao_linear(modelo, nomes_colunas):
    coeficientes = modelo.coef_
    intercepto = modelo.intercept_
    df_params = pd.DataFrame({'Variável': nomes_colunas, 'Peso': coeficientes})
    df_params['Abs'] = df_params['Peso'].abs()
    print("\n=== PESOS DO MODELO DE VOLUME ===")
    print(df_params.sort_values(by='Abs', ascending=False).drop(columns='Abs'))
    print("-" * 30)

# --- 3. LOOP DE VALIDAÇÃO ---

tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados = []

print("Iniciando Validação Cruzada com CLIMATOLOGIA NA VAZÃO...\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]
    
    print(f"Fold {fold+1}: Teste de {df_test.index[0].date()} a {df_test.index[-1].date()}")

    # =================================================================
    # ETAPA 1: PREPARAÇÃO (Usando dados de TREINO)
    # =================================================================
    
    # A. Calcular Climatologia (A "Previsão" da Vazão)
    # Baseada apenas no histórico disponível até aquele momento (df_train)
    curva_climatologia = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural, window_days=7)
    
    # B. Treinar Modelo de Volume (Com Vazão Real)
    model_volume = LinearRegression()
    model_volume, X_full_eq = treinar_modelo_volume(
        df_train, nome_coluna_volume, nome_coluna_vazao_natural, model_volume
    )
    if fold == 0: mostrar_equacao_linear(model_volume, X_full_eq.columns)

    # =================================================================
    # ETAPA 2: PREVISÃO RECURSIVA DO VOLUME
    # =================================================================
    
    # Prepara features determinísticas de volume para o futuro (datas de teste)
    X_trend_test_v = criar_features_deterministicas_volume(df_test.index)
    
    # Buffer para cálculo de lags e derivadas do volume
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    
    preds_volume = []
    preds_vn_utilizada = [] # Para guardar qual vazão (climatologia) foi usada no dia
    
    # LOOP DIA A DIA
    for i in range(len(df_test)):
        current_date = df_test.index[i]
        idx_atual = [current_date]
        
        # --- 1. Obter Vazão Natural (Via Climatologia) ---
        day_of_year = current_date.dayofyear
        # Se for bissexto (366) e a curva foi feita sem tratar, usamos 365 ou tratamos
        if day_of_year > 366: day_of_year = 366 
        
        # O "chute" do modelo para a chuva de hoje é a média histórica
        try:
            vazao_n_input = curva_climatologia.loc[day_of_year]
        except KeyError:
             # Fallback caso dia 366 dê erro
            vazao_n_input = curva_climatologia.iloc[-1]
            
        preds_vn_utilizada.append(vazao_n_input)
        
        # --- 2. Prever Volume ---
        # Recupera valores do buffer (Volume previsto dias anteriores)
        vol_t_1 = buffer_vols[-1] 
        vol_t_2 = buffer_vols[-2] 
        vol_t_3 = buffer_vols[-3] 
        
        # Calcula derivadas
        feat_diff = vol_t_1 - vol_t_2
        feat_diff2 = (vol_t_1 - vol_t_2) - (vol_t_2 - vol_t_3)
        
        # Monta DataFrame Dinâmico
        df_dynamic = pd.DataFrame({
            'vol_lag1':  [vol_t_1],
            'vol_diff':  [feat_diff],
            'vol_diff2': [feat_diff2],
            'vazao_n':   [vazao_n_input] # Aqui entra a CLIMATOLOGIA
        }, index=idx_atual)
        
        # Pega a parte estática (sazonalidade)
        df_static = X_trend_test_v.iloc[[i]]
        
        # Concatena e prevê
        X_vol_input = pd.concat([df_dynamic, df_static], axis=1)
        
        vol_pred = model_volume.predict(X_vol_input)[0]
        preds_volume.append(vol_pred)
        
        # Atualiza Buffer
        buffer_vols.append(vol_pred)
        buffer_vols.pop(0)

    # =================================================================
    # ETAPA 3: AVALIAÇÃO
    # =================================================================
    
    y_true_vol = df_test[nome_coluna_volume]
    y_true_vn = df_test[nome_coluna_vazao_natural]
    
    rmse_vol = np.sqrt(mean_squared_error(y_true_vol, preds_volume))
    rmse_vn_clim = np.sqrt(mean_squared_error(y_true_vn, preds_vn_utilizada))
    
    resultados.append(rmse_vol)
    
    print(f"--- RESULTADOS FOLD {fold+1} ---")
    print(f"[VOLUME] RMSE: {rmse_vol:.4f}")
    print(f"[VAZÃO - CLIMATOLOGIA] RMSE: {rmse_vn_clim:.4f}")
    
    # PLOTAGEM
    fig, axs = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Gráfico 1: Volume
    axs[0].plot(y_true_vol.index, y_true_vol, label='Real', color='navy')
    axs[0].plot(y_true_vol.index, preds_volume, label='Previsto (c/ Climatologia)', color='darkorange', linestyle='--')
    axs[0].set_title(f'Fold {fold+1} - Volume (RMSE: {rmse_vol:.2f})')
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)
    
    # Gráfico 2: Vazão (Real vs Climatologia usada)
    axs[1].plot(y_true_vn.index, y_true_vn, label='Vazão Real (Chuva)', color='blue', alpha=0.4)
    axs[1].plot(y_true_vn.index, preds_vn_utilizada, label='Climatologia Usada', color='green', linewidth=2)
    axs[1].set_title(f'Input Exógeno: Climatologia vs Real (RMSE: {rmse_vn_clim:.2f})')
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio (Volume): {np.mean(resultados):.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
VALIDATION_SIZE = 90
LAGS_VAZAO = 2  # Lags para o modelo ML de vazão
LAGS_VOLUME = 1

# Supondo df já carregado
# df = ... 

# --- 1. FUNÇÕES: CLIMATOLOGIA (Feature 1) ---
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        # Filtro simples de dias
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            # Remove outliers (10% - 90%)
            vals = vals[(vals >= vals.quantile(0.10)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature(index, curva_clim):
    """Retorna array com valores da climatologia para as datas do index"""
    days = index.dayofyear
    # Ajuste para bissexto > 366 (se houver erro, usa o ultimo valor)
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

# --- 2. FUNÇÕES: MODELO ML DE VAZÃO (Feature 2) ---
def criar_feat_fourier(index):
    dp = DeterministicProcess(index=index, constant=True, order=1, 
                              additional_terms=[CalendarFourier("YE", 2)], drop=True)
    return dp.in_sample()

def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for l in range(1, lags+1):
        df_lags[f'lag_{l}'] = series.shift(l)
    return df_lags

def treinar_modelo_vazao_ml(df_train, col_target):
    # 1. Tendência
    y = df_train[col_target]
    X_trend = criar_feat_fourier(df_train.index)
    
    # Modelo de Tendência (Lasso para ser mais robusto)
    m_trend = Ridge(alpha=0.5)
    m_trend.fit(X_trend, y)
    trend_pred = pd.Series(m_trend.predict(X_trend), index=y.index)
    
    # 2. Resíduos
    resid = y - trend_pred
    X_resid = criar_lags(resid, LAGS_VAZAO).dropna()
    y_resid = resid.loc[X_resid.index]
    
    # Modelo de Resíduos (KNN ou RF)
    m_resid = KNeighborsRegressor(n_neighbors=10) # Aumentei vizinhos para suavizar
    m_resid.fit(X_resid, y_resid)
    
    last_resids = resid.tail(LAGS_VAZAO).values
    return m_trend, m_resid, last_resids

# --- 3. FUNÇÃO: MODELO DE VOLUME (O "JUIZ") ---
def treinar_modelo_volume_final(df_train, col_vol, col_vn, curva_clim, m_trend_vn, m_resid_vn):
    """
    Treina o volume usando:
    1. Lags do Volume
    2. Feature Vazão Climatologia
    3. Feature Vazão ML (In-Sample Prediction)
    """
    y = df_train[col_vol]
    
    # --- Engenharia de Features Dinâmicas ---
    X = pd.DataFrame(index=df_train.index)
    X['vol_lag1'] = df_train[col_vol].shift(1)
    X['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # --- Feature Exógena 1: Climatologia ---
    X['feat_vazao_clim'] = get_clim_feature(df_train.index, curva_clim)
    
    # --- Feature Exógena 2: Modelo ML (Predição In-Sample) ---
    # Recriamos a previsão que o modelo ML faria no treino para o Volume aprender a confiar (ou não) nele
    X_trend = criar_feat_fourier(df_train.index)
    trend_in = m_trend_vn.predict(X_trend)
    
    # Resíduos in-sample (aproximado via lags reais para treinar rápido)
    resid_real = df_train[col_vn] - trend_in
    X_resid_lags = criar_lags(resid_real, LAGS_VAZAO) # Lags reais
    # Para simplificar e evitar vazamento total, vamos usar a Vazão Real como proxy do "Melhor ML Possível"
    # ou usar a predição real. Vamos usar a predição in-sample do modelo:
    resid_pred_in = m_resid_vn.predict(X_resid_lags.fillna(0)) 
    
    X['feat_vazao_ml'] = trend_in + resid_pred_in
    
    # Drop NaNs gerados pelos lags
    X_full = X.dropna()
    y_train = y.loc[X_full.index]
    
    model_vol = LinearRegression()
    model_vol.fit(X_full, y_train)
    
    return model_vol, X_full.columns

# --- 4. LOOP DE VALIDAÇÃO ---
tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados = []

print("=== INICIANDO MODELO HÍBRIDO DUPLO (CLIM + ML) ===\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    
    # A. Treinar Climatologia
    curva_clim = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural)
    
    # B. Treinar Modelo Vazão ML
    m_vn_trend, m_vn_resid, last_resids_vn = treinar_modelo_vazao_ml(df_train, nome_coluna_vazao_natural)
    
    # C. Treinar Modelo Volume (Com as 2 features)
    model_vol, col_names = treinar_modelo_volume_final(
        df_train, nome_coluna_volume, nome_coluna_vazao_natural, 
        curva_clim, m_vn_trend, m_vn_resid
    )
    
    # Mostrar pesos no primeiro fold
    if fold == 0:
        coefs = pd.Series(model_vol.coef_, index=col_names)
        print("--- PESOS QUE O MODELO DE VOLUME DEU ---")
        print(coefs.sort_values(ascending=False))
        print("Note: Se 'feat_vazao_clim' for maior que 'feat_vazao_ml', o modelo confia mais na média histórica.\n")

    # --- PREVISÃO RECURSIVA ---
    # Preparação
    X_trend_test_vn = criar_feat_fourier(df_test.index)
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    hist_resid_vn = list(last_resids_vn)
    
    preds_vol = []
    
    # Listas para guardar as features usadas (para plotar depois)
    log_feat_clim = []
    log_feat_ml = []
    
    for i in range(len(df_test)):
        date = df_test.index[i]
        
        # 1. Feature Climatologia
        val_clim = get_clim_feature(pd.DatetimeIndex([date]), curva_clim)[0]
        
        # 2. Feature ML Recursivo
        # a. Trend
        val_trend = m_vn_trend.predict(X_trend_test_vn.iloc[[i]])[0]
        # b. Resid
        lags_resid = np.array(hist_resid_vn[-LAGS_VAZAO:][::-1])
        val_resid = m_vn_resid.predict(pd.DataFrame([lags_resid], columns=[f'lag_{k}' for k in range(1,LAGS_VAZAO+1)]))[0]
        
        val_ml = val_trend + val_resid
        if val_ml < 0: val_ml = 0 # Trava física
        
        # Atualiza histórico de resíduo do ML
        hist_resid_vn.append(val_resid)
        
        # Logs para plot
        log_feat_clim.append(val_clim)
        log_feat_ml.append(val_ml)
        
        # 3. Prever Volume
        vol_t1 = buffer_vols[-1]
        vol_diff = vol_t1 - buffer_vols[-2]
        
        # Monta input para o modelo de volume (MESMA ORDEM DO TREINO)
        # Colunas: ['vol_lag1', 'vol_diff', 'feat_vazao_clim', 'feat_vazao_ml']
        input_vol = pd.DataFrame({
            'vol_lag1': [vol_t1],
            'vol_diff': [vol_diff],
            'feat_vazao_clim': [val_clim],
            'feat_vazao_ml': [val_ml]
        })
        
        pred_vol = model_vol.predict(input_vol)[0]
        preds_vol.append(pred_vol)
        
        # Atualiza buffer volume
        buffer_vols.append(pred_vol)
        buffer_vols.pop(0)

    # --- AVALIAÇÃO ---
    rmse = np.sqrt(mean_squared_error(df_test[nome_coluna_volume], preds_vol))
    resultados.append(rmse)
    
    print(f"Fold {fold+1} RMSE: {rmse:.4f}")
    
    # Plot Diagnóstico
    fig, ax = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Gráfico 1: Volume
    ax[0].plot(df_test.index, df_test[nome_coluna_volume], label='Real', color='navy')
    ax[0].plot(df_test.index, preds_vol, label='Previsto Híbrido', color='orange', linestyle='--')
    ax[0].set_title(f"Volume - Fold {fold+1}")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Gráfico 2: Batalha das Features de Vazão
    ax[1].plot(df_test.index, df_test[nome_coluna_vazao_natural], label='Vazão REAL (Alvo)', color='black', alpha=0.3, linewidth=3)
    ax[1].plot(df_test.index, log_feat_clim, label='Feature: Climatologia', color='green')
    ax[1].plot(df_test.index, log_feat_ml, label='Feature: Modelo ML', color='red', linestyle=':')
    ax[1].set_title("Inputs Exógenos: Climatologia vs ML")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio Final: {np.mean(resultados):.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
VALIDATION_SIZE = 90
LAGS_RESIDUO = 2  # Lags do "Ciclo" (desvio da média)
LAGS_VOLUME = 1

# Supondo df já carregado
# df = ... 
# Certifique-se que o índice é Datetime

# --- 1. FUNÇÕES: CLIMATOLOGIA (A BASE) ---
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            # Remove outliers (10% - 90%) para pegar o "padrão normal"
            vals = vals[(vals >= vals.quantile(0.10)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature(index, curva_clim):
    """Retorna array com valores da climatologia para as datas do index"""
    days = index.dayofyear
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

# --- 2. FUNÇÕES: MODELO DE CICLO (ML NOS RESÍDUOS) ---
def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for l in range(1, lags+1):
        df_lags[f'lag_{l}'] = series.shift(l)
    return df_lags

def treinar_modelo_ciclo(df_train, col_target, curva_clim):
    """
    1. Calcula o erro da climatologia no treino (Real - Climatologia).
    2. Treina um ML para prever esse erro (Ciclo) baseado nos erros passados.
    """
    y_real = df_train[col_target]
    y_clim = get_clim_feature(df_train.index, curva_clim)
    
    # O Alvo do ML é o RESÍDUO (O quanto a climatologia errou)
    residuo = y_real - y_clim 
    
    # Features: Lags do próprio resíduo (Autocorrelação do erro)
    # Ex: Se ontem choveu muito acima da média, hoje tende a chover acima também?
    X_lags = criar_lags(residuo, LAGS_RESIDUO).dropna()
    y_target_resid = residuo.loc[X_lags.index]
    
    # Modelo ML (Random Forest é bom para capturar padrões não lineares no erro)
    # Pode usar KNN também se quiser algo mais suave
    model_resid = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    model_resid.fit(X_lags, y_target_resid)
    
    # Retorna o modelo e os últimos resíduos para iniciar o loop de teste
    last_resids = residuo.tail(LAGS_RESIDUO).values
    return model_resid, last_resids

# --- 3. FUNÇÃO: MODELO DE VOLUME ---
def treinar_modelo_volume(df_train, col_vol, col_vn):
    """
    Treina o modelo físico de Volume.
    Usamos a VAZÃO REAL no treino para ele aprender a física correta:
    (Entrou Água -> Volume Subiu).
    """
    y = df_train[col_vol]
    
    X = pd.DataFrame(index=df_train.index)
    X['vol_lag1'] = df_train[col_vol].shift(1)
    X['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # Feature Exógena: Vazão Natural (Aqui usamos a REAL para aprender a relação física)
    X['vazao_input'] = df_train[col_vn]
    
    X_full = X.dropna()
    y_train = y.loc[X_full.index]
    
    model_vol = LinearRegression()
    model_vol.fit(X_full, y_train)
    
    return model_vol

# --- 4. LOOP DE VALIDAÇÃO ---
tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados_vol = []
resultados_vazao = []

print("=== INICIANDO MODELO: CLIMATOLOGIA (BASE) + ML (CICLO) ===\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    
    # --- ETAPA A: TREINAMENTO ---
    
    # 1. Calcula Climatologia (Base)
    curva_clim = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural)
    
    # 2. Treina ML para prever o Ciclo (Resíduo da Climatologia)
    m_ciclo, last_resids_cycle = treinar_modelo_ciclo(df_train, nome_coluna_vazao_natural, curva_clim)
    
    # 3. Treina Volume (Aprende a relação Volume <-> Vazão)
    m_volume = treinar_modelo_volume(df_train, nome_coluna_volume, nome_coluna_vazao_natural)
    
    # --- ETAPA B: PREVISÃO RECURSIVA ---
    
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    hist_resid_cycle = list(last_resids_cycle)
    
    preds_vol = []
    preds_vazao_construida = [] # Para avaliarmos se essa construção ficou boa
    
    for i in range(len(df_test)):
        date = df_test.index[i]
        
        # 1. PREVISÃO DA VAZÃO (HÍBRIDA)
        # Parte 1: Base Climatológica
        val_base = get_clim_feature(pd.DatetimeIndex([date]), curva_clim)[0]
        
        # Parte 2: Ciclo (ML prevê o desvio baseado nos desvios anteriores)
        lags_cycle = np.array(hist_resid_cycle[-LAGS_RESIDUO:][::-1])
        # Reshape para (1, n_features)
        pred_desvio = m_ciclo.predict(pd.DataFrame([lags_cycle], columns=[f'lag_{k}' for k in range(1,LAGS_RESIDUO+1)]))[0]
        
        # Vazão Final = Base + Desvio
        vazao_final = val_base + pred_desvio
        
        # Trava física: Não existe vazão negativa, nem desvio que zere absurdamente se não for seca
        if vazao_final < 0: vazao_final = 0
        
        preds_vazao_construida.append(vazao_final)
        
        # Atualiza o histórico de resíduos com o DESVIO PREVISTO (Recursão)
        hist_resid_cycle.append(pred_desvio)
        
        # 2. PREVISÃO DO VOLUME
        vol_t1 = buffer_vols[-1]
        vol_diff = vol_t1 - buffer_vols[-2]
        
        input_vol = pd.DataFrame({
            'vol_lag1': [vol_t1],
            'vol_diff': [vol_diff],
            'vazao_input': [vazao_final] # Entra a vazão construída
        })
        
        pred_vol = m_volume.predict(input_vol)[0]
        preds_vol.append(pred_vol)
        
        buffer_vols.append(pred_vol)
        buffer_vols.pop(0)

    # --- AVALIAÇÃO ---
    rmse_vol = np.sqrt(mean_squared_error(df_test[nome_coluna_volume], preds_vol))
    rmse_vazao = np.sqrt(mean_squared_error(df_test[nome_coluna_vazao_natural], preds_vazao_construida))
    
    resultados_vol.append(rmse_vol)
    resultados_vazao.append(rmse_vazao)
    
    print(f"Fold {fold+1} | RMSE Volume: {rmse_vol:.4f} | RMSE Vazão (Híbrida): {rmse_vazao:.4f}")
    
    # --- PLOT DETALHADO ---
    fig, ax = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Gráfico 1: O Resultado Final (Volume)
    ax[0].plot(df_test.index, df_test[nome_coluna_volume], label='Volume Real', color='navy')
    ax[0].plot(df_test.index, preds_vol, label='Volume Previsto', color='darkorange', linestyle='--')
    ax[0].set_title(f"Previsão de Volume - Fold {fold+1}")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Gráfico 2: Como construímos a vazão?
    ax[1].plot(df_test.index, df_test[nome_coluna_vazao_natural], label='Vazão Real', color='black', alpha=0.3)
    
    # Plota a Climatologia pura (para vermos a base)
    clim_pura = get_clim_feature(df_test.index, curva_clim)
    ax[1].plot(df_test.index, clim_pura, label='Base (Climatologia)', color='green', linestyle=':', linewidth=1)
    
    # Plota a Vazão Final (Base + Ciclo)
    ax[1].plot(df_test.index, preds_vazao_construida, label='Híbrido (Base + Ciclo ML)', color='red', linewidth=1.5)
    
    ax[1].set_title(f"Decomposição da Vazão (RMSE: {rmse_vazao:.2f})")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio Final (Volume): {np.mean(resultados_vol):.4f}")
print(f"RMSE Médio Final (Vazão): {np.mean(resultados_vazao):.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
VALIDATION_SIZE = 90
LAGS_CICLO = 2  # Lags do resíduo
LAGS_VOLUME = 1

# Supondo df já carregado no ambiente
# df = ... 

# ==============================================================================
# 1. CLIMATOLOGIA (A BASE DE SOMA)
# ==============================================================================
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            # Remove outliers extremos
            vals = vals[(vals >= vals.quantile(0.10)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature(index, curva_clim):
    days = index.dayofyear
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

# ==============================================================================
# 2. MODELO MATEMÁTICO (PARA EXTRAIR O CICLO/RESÍDUO)
# ==============================================================================
def criar_feat_fourier(index):
    # Cria a base matemática (Tendência Linear + Sazonalidade Perfeita)
    dp = DeterministicProcess(
        index=index, 
        constant=True, 
        order=1, # Tendência Linear
        seasonal=False, 
        additional_terms=[CalendarFourier("YE", 2)], # Fourier ordem 2
        drop=True
    )
    return dp.in_sample()

def treinar_modelo_extracao_residuo(df_train, col_target):
    """
    Treina um modelo Linear/Ridge apenas para capturar a tendência e sazonalidade matemática.
    O objetivo é gerar o resíduo: Real - Matemático.
    """
    y = df_train[col_target]
    X = criar_feat_fourier(df_train.index)
    
    model_math = LinearRegression()
    # model_math = Ridge(alpha=0.5)
    model_math.fit(X, y)
    
    # Gera a curva matemática ("Limpa")
    y_math_clean = pd.Series(model_math.predict(X), index=y.index)
    
    # O CICLO é o que sobrou (Resíduo)
    residuo_ciclo = y - y_math_clean
    
    return model_math, residuo_ciclo

# ==============================================================================
# 3. MODELO ML (APRENDE O PADRÃO DO CICLO)
# ==============================================================================
def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for l in range(1, lags+1):
        df_lags[f'lag_{l}'] = series.shift(l)
    return df_lags

def treinar_modelo_ciclo_ml(residuo_series):
    """
    Treina o ML para prever o próximo resíduo baseados nos anteriores.
    """
    X_lags = criar_lags(residuo_series, LAGS_CICLO).dropna()
    y_target = residuo_series.loc[X_lags.index]
    
    # Random Forest captura bem a não-linearidade do ruído
    model_ml = KNeighborsRegressor(n_neighbors=5)
    model_ml.fit(X_lags, y_target)
    
    last_resids = residuo_series.tail(LAGS_CICLO).values
    return model_ml, last_resids

# ==============================================================================
# 4. MODELO DE VOLUME (FÍSICO)
# ==============================================================================
def treinar_modelo_volume(df_train, col_vol, col_vn):
    y = df_train[col_vol]
    X = pd.DataFrame(index=df_train.index)
    X['vol_lag1'] = df_train[col_vol].shift(1)
    X['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    X['vazao_input'] = df_train[col_vn] # Treina com a Real
    
    X_full = X.dropna()
    y_train = y.loc[X_full.index]
    
    model_vol = LinearRegression()
    model_vol.fit(X_full, y_train)
    return model_vol

# ==============================================================================
# 5. LOOP DE VALIDAÇÃO
# ==============================================================================
tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados_vol = []
resultados_vazao = []

print("=== INICIANDO MODELO: RESÍDUO DE FOURIER + BASE CLIMATOLÓGICA ===\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    
    # --- A. TREINAMENTO ---
    
    # 1. Base Climatológica (Calculada no Treino)
    curva_clim = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural)
    
    # 2. Extração do Ciclo (Usando Modelo Matemático Fourier)
    model_math, serie_residuos_treino = treinar_modelo_extracao_residuo(df_train, nome_coluna_vazao_natural)
    
    # 3. Treino do ML no Ciclo
    model_ml_ciclo, last_resids_cycle = treinar_modelo_ciclo_ml(serie_residuos_treino)
    
    # 4. Treino do Volume
    model_vol = treinar_modelo_volume(df_train, nome_coluna_volume, nome_coluna_vazao_natural)
    
    # --- B. PREVISÃO RECURSIVA ---
    
    # Buffer para volume
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    # Histórico para o ciclo (inicializado com os últimos resíduos REAIS do treino)
    hist_resid_cycle = list(last_resids_cycle)
    
    preds_vol = []
    preds_vazao_hibrida = [] 
    
    for i in range(len(df_test)):
        date = df_test.index[i]
        
        # 1. CALCULAR VAZÃO
        
        # A) Pega a Base (Climatologia)
        val_base_clim = get_clim_feature(pd.DatetimeIndex([date]), curva_clim)[0]
        
        # B) Prevê o Ciclo (Usando o ML treinado nos resíduos do Fourier)
        lags_cycle = np.array(hist_resid_cycle[-LAGS_CICLO:][::-1])
        cycle_pred = model_ml_ciclo.predict(pd.DataFrame([lags_cycle], columns=[f'lag_{k}' for k in range(1,LAGS_CICLO+1)]))[0]
        
        # C) SOMA HÍBRIDA: Climatologia + Ciclo(Fourier)
        vazao_final = val_base_clim + cycle_pred
        
        if vazao_final < 0: vazao_final = 0
        preds_vazao_hibrida.append(vazao_final)
        
        # Atualiza histórico de ciclo com o previsto (Recursão)
        hist_resid_cycle.append(cycle_pred)
        
        # 2. CALCULAR VOLUME
        vol_t1 = buffer_vols[-1]
        vol_diff = vol_t1 - buffer_vols[-2]
        
        input_vol = pd.DataFrame({
            'vol_lag1': [vol_t1],
            'vol_diff': [vol_diff],
            'vazao_input': [vazao_final]
        })
        
        pred_vol = model_vol.predict(input_vol)[0]
        preds_vol.append(pred_vol)
        
        buffer_vols.append(pred_vol)
        buffer_vols.pop(0)

    # --- AVALIAÇÃO ---
    rmse_vol = np.sqrt(mean_squared_error(df_test[nome_coluna_volume], preds_vol))
    rmse_vazao = np.sqrt(mean_squared_error(df_test[nome_coluna_vazao_natural], preds_vazao_hibrida))
    
    resultados_vol.append(rmse_vol)
    
    print(f"Fold {fold+1} | RMSE Volume: {rmse_vol:.4f} | RMSE Vazão: {rmse_vazao:.4f}")
    
    # PLOT
    fig, ax = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Gráfico Volume
    ax[0].plot(df_test.index, df_test[nome_coluna_volume], label='Volume Real', color='navy')
    ax[0].plot(df_test.index, preds_vol, label='Volume Previsto', color='darkorange', linestyle='--')
    ax[0].set_title(f"Fold {fold+1} - Volume")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Gráfico Vazão (O Pulo do Gato)
    ax[1].plot(df_test.index, df_test[nome_coluna_vazao_natural], label='Real', color='black', alpha=0.3)
    
    # Mostra a base Climatológica
    clim_base = get_clim_feature(df_test.index, curva_clim)
    ax[1].plot(df_test.index, clim_base, label='Base (Climatologia)', color='green', linestyle=':', linewidth=1)
    
    # Mostra o resultado final
    ax[1].plot(df_test.index, preds_vazao_hibrida, label='Final (Clim + Ciclo ML)', color='red', linewidth=1.5)
    
    ax[1].set_title("Composição da Vazão: Base Climatológica + Ciclo Aprendido via Fourier")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio (Volume): {np.mean(resultados_vol):.4f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
VALIDATION_SIZE = 90
LAGS_CICLO = 2  # Lags do resíduo da vazão
LAGS_VOLUME = 1

# Supondo df já carregado no ambiente
# df = ... 

# ==============================================================================
# 1. CLIMATOLOGIA (A BASE DE TUDO)
# ==============================================================================
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            # Remove outliers extremos (10% - 90%)
            vals = vals[(vals >= vals.quantile(0.10)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature(index, curva_clim):
    """Retorna array com valores da climatologia para as datas do index"""
    days = index.dayofyear
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

# ==============================================================================
# 2. MODELO MATEMÁTICO (PARA EXTRAIR O CICLO DA VAZÃO)
# ==============================================================================
def criar_feat_fourier(index):
    dp = DeterministicProcess(
        index=index, 
        constant=True, 
        order=1, 
        seasonal=False, 
        additional_terms=[CalendarFourier("YE", 2)], 
        drop=True
    )
    return dp.in_sample()

def treinar_modelo_extracao_residuo(df_train, col_target):
    y = df_train[col_target]
    X = criar_feat_fourier(df_train.index)
    
    model_math = Ridge(alpha = 0.5)
    model_math.fit(X, y)
    
    # Curva matemática limpa
    y_math_clean = pd.Series(model_math.predict(X), index=y.index)
    
    # Resíduo (Ciclo)
    residuo_ciclo = y - y_math_clean
    return model_math, residuo_ciclo

# ==============================================================================
# 3. MODELO ML (APRENDE O PADRÃO DO CICLO DA VAZÃO)
# ==============================================================================
def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for l in range(1, lags+1):
        df_lags[f'lag_{l}'] = series.shift(l)
    return df_lags

def treinar_modelo_ciclo_ml(residuo_series):
    X_lags = criar_lags(residuo_series, LAGS_CICLO).dropna()
    y_target = residuo_series.loc[X_lags.index]
    
    # KNN captura padrões locais de repetição do erro
    model_ml = KNeighborsRegressor(n_neighbors=5)
    model_ml.fit(X_lags, y_target)
    
    last_resids = residuo_series.tail(LAGS_CICLO).values
    return model_ml, last_resids

# ==============================================================================
# 4. MODELO DE VOLUME (AGORA COM CLIMATOLOGIA DO VOLUME)
# ==============================================================================
def treinar_modelo_volume(df_train, col_vol, col_vn, curva_clim_vol):
    y = df_train[col_vol]
    X = pd.DataFrame(index=df_train.index)
    
    # Features Autoregressivas
    X['vol_lag1'] = df_train[col_vol].shift(1)
    X['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # Feature Exógena 1: Vazão Real
    X['vazao_input'] = df_train[col_vn] 
    
    # Feature Exógena 2: Climatologia do Volume (O ÂNCORA)
    X['vol_clim'] = get_clim_feature(df_train.index, curva_clim_vol)
    
    X_full = X.dropna()
    y_train = y.loc[X_full.index]
    
    model_vol = LinearRegression()
    model_vol.fit(X_full, y_train)
    
    # Print dos coeficientes para conferirmos a importância
    print(f"\n--- Pesos do Modelo de Volume ---")
    coefs = pd.Series(model_vol.coef_, index=X_full.columns)
    print(coefs.sort_values(ascending=False))
    print("-" * 30)
    
    return model_vol

# ==============================================================================
# 5. LOOP DE VALIDAÇÃO
# ==============================================================================
tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados_vol = []

print("=== INICIANDO MODELO FINAL: Vol(Lags + VazãoHybrid + VolClim) ===\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    
    # --- A. TREINAMENTO ---
    
    # 1. Calcular Climatologias (Vazão e Volume)
    curva_clim_vazao = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural)
    curva_clim_volume = calcular_climatologia_robusta(df_train, nome_coluna_volume) # NOVA
    
    # 2. Pipeline Vazão (Extração Ciclo -> ML)
    model_math, serie_residuos_treino = treinar_modelo_extracao_residuo(df_train, nome_coluna_vazao_natural)
    model_ml_ciclo, last_resids_cycle = treinar_modelo_ciclo_ml(serie_residuos_treino)
    
    # 3. Treino do Volume (Com a nova feature de Climatologia Volume)
    model_vol = treinar_modelo_volume(df_train, nome_coluna_volume, nome_coluna_vazao_natural, curva_clim_volume)
    
    # --- B. PREVISÃO RECURSIVA ---
    
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    hist_resid_cycle = list(last_resids_cycle)
    
    preds_vol = []
    
    for i in range(len(df_test)):
        date = df_test.index[i]
        
        # --- PASSO 1: CALCULAR VAZÃO HÍBRIDA ---
        # Base (Climatologia Vazão)
        val_base_vazao = get_clim_feature(pd.DatetimeIndex([date]), curva_clim_vazao)[0]
        
        # Ciclo (ML sobre resíduo Fourier)
        lags_cycle = np.array(hist_resid_cycle[-LAGS_CICLO:][::-1])
        cycle_pred = model_ml_ciclo.predict(pd.DataFrame([lags_cycle], columns=[f'lag_{k}' for k in range(1,LAGS_CICLO+1)]))[0]
        
        vazao_final = val_base_vazao + cycle_pred
        if vazao_final < 0: vazao_final = 0
        
        # Atualiza histórico ciclo
        hist_resid_cycle.append(cycle_pred)
        
        # --- PASSO 2: CALCULAR VOLUME ---
        vol_t1 = buffer_vols[-1]
        vol_diff = vol_t1 - buffer_vols[-2]
        
        # Nova Feature: Climatologia do Volume para hoje
        vol_clim_hoje = get_clim_feature(pd.DatetimeIndex([date]), curva_clim_volume)[0]
        
        input_vol = pd.DataFrame({
            'vol_lag1': [vol_t1],
            'vol_diff': [vol_diff],
            'vazao_input': [vazao_final],
            'vol_clim': [vol_clim_hoje] # A âncora entra aqui
        })
        
        pred_vol = model_vol.predict(input_vol)[0]
        preds_vol.append(pred_vol)
        
        buffer_vols.append(pred_vol)
        buffer_vols.pop(0)

    # --- AVALIAÇÃO ---
    rmse_vol = np.sqrt(mean_squared_error(df_test[nome_coluna_volume], preds_vol))
    resultados_vol.append(rmse_vol)
    
    print(f"Fold {fold+1} | RMSE Volume: {rmse_vol:.4f}")
    
    # PLOT
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(df_test.index, df_test[nome_coluna_volume], label='Real', color='navy', linewidth=2)
    ax.plot(df_test.index, preds_vol, label='Previsto', color='darkorange', linestyle='--', linewidth=2)
    
    # Plota a "Âncora" (Climatologia Volume) para ver se o modelo está orbitando ela
    vol_clim_test = get_clim_feature(df_test.index, curva_clim_volume)
    ax.plot(df_test.index, vol_clim_test, label='Climatologia (Âncora)', color='green', linestyle=':', alpha=0.6)
    
    ax.set_title(f"Fold {fold+1} - Previsão Volume (RMSE: {rmse_vol:.2f})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio (Volume): {np.mean(resultados_vol):.4f}")

# MELHOR

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler # <--- IMPORTANTE
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
VALIDATION_SIZE = 90
LAGS_CICLO = 2  # Lags do resíduo
LAGS_VOLUME = 1

# Supondo df já carregado no ambiente
# df = ... 
df = import_dataframe()
#df[df['Data']>='2018-01-01']
df.set_index("Data", inplace=True)

# ==============================================================================
# 1. CLIMATOLOGIA (A BASE DE SOMA)
# ==============================================================================
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            # Remove outliers extremos
            vals = vals[(vals >= vals.quantile(0.1)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature(index, curva_clim):
    days = index.dayofyear
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

# ==============================================================================
# 2. MODELO MATEMÁTICO (PARA EXTRAIR O CICLO/RESÍDUO)
# ==============================================================================
def criar_feat_fourier(index):
    # Cria a base matemática (Tendência Linear + Sazonalidade Perfeita)
    dp = DeterministicProcess(
        index=index, 
        constant=True, 
        order=1, # Tendência Linear
        seasonal=False, 
        additional_terms=[CalendarFourier("YE", 2)], # Fourier ordem 2
        drop=True
    )
    return dp.in_sample()

def treinar_modelo_extracao_residuo(df_train, col_target):
    """
    Treina um modelo Linear/Ridge apenas para capturar a tendência e sazonalidade matemática.
    O objetivo é gerar o resíduo: Real - Matemático.
    """
    y = df_train[col_target]
    X = criar_feat_fourier(df_train.index)
    
    model_math = LinearRegression()
    # model_math = Ridge(alpha=0.5)
    model_math.fit(X, y)
    
    # Gera a curva matemática ("Limpa")
    y_math_clean = pd.Series(model_math.predict(X), index=y.index)
    
    # O CICLO é o que sobrou (Resíduo)
    residuo_ciclo = y - y_math_clean
    
    return model_math, residuo_ciclo

# ==============================================================================
# 3. MODELO ML (APRENDE O PADRÃO DO CICLO)
# ==============================================================================
def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for l in range(1, lags+1):
        df_lags[f'lag_{l}'] = series.shift(l)
    return df_lags

def treinar_modelo_ciclo_ml(residuo_series):
    """
    Treina o ML para prever o próximo resíduo baseados nos anteriores.
    """
    X_lags = criar_lags(residuo_series, LAGS_CICLO).dropna()
    y_target = residuo_series.loc[X_lags.index]
    
    # Random Forest captura bem a não-linearidade do ruído
    model_ml = KNeighborsRegressor(n_neighbors=5)
    model_ml.fit(X_lags, y_target)
    
    last_resids = residuo_series.tail(LAGS_CICLO).values
    return model_ml, last_resids

# ==============================================================================
# 4. MODELO DE VOLUME (COM SCALING)
# ==============================================================================
def treinar_modelo_volume(df_train, col_vol, col_vn):
    y = df_train[col_vol]
    X = pd.DataFrame(index=df_train.index)
    X['vol_lag1'] = df_train[col_vol].shift(1)
    X['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    X['vazao_input'] = df_train[col_vn] # Treina com a Real
    
    X_full = X.dropna()
    y_train = y.loc[X_full.index]
    
    # --- APLICANDO O SCALER ---
    scaler = StandardScaler()
    # O fit_transform calcula Média e Desvio Padrão do TREINO e já aplica
    X_scaled = scaler.fit_transform(X_full)
    
    # Para manter como DataFrame (opcional, mas bom para debug de colunas)
    X_scaled_df = pd.DataFrame(X_scaled, columns=X_full.columns, index=X_full.index)
    
    model_vol = LinearRegression()
    model_vol.fit(X_scaled_df, y_train)
    
    # Retorna o modelo e o SCALER (precisaremos dele no teste)
    return model_vol, scaler

# ==============================================================================
# 5. LOOP DE VALIDAÇÃO
# ==============================================================================
tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados_vol = []
resultados_vazao = []

print("=== INICIANDO MODELO: RESÍDUO DE FOURIER + BASE CLIMATOLÓGICA (COM SCALING) ===\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    
    # --- A. TREINAMENTO ---
    
    # 1. Base Climatológica (Calculada no Treino)
    curva_clim = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural)
    
    # 2. Extração do Ciclo (Usando Modelo Matemático Fourier)
    model_math, serie_residuos_treino = treinar_modelo_extracao_residuo(df_train, nome_coluna_vazao_natural)
    # modelo sazonalidade + tendencia -> ytrue - ypred -> residuo -> outro modelo e depois somava os dois y_pred_resid
    # 3. Treino do ML no Ciclo
    model_ml_ciclo, last_resids_cycle = treinar_modelo_ciclo_ml(serie_residuos_treino)
    
    # 4. Treino do Volume (Retorna Modelo + Scaler)
    model_vol, scaler_vol = treinar_modelo_volume(df_train, nome_coluna_volume, nome_coluna_vazao_natural)
    
    # --- B. PREVISÃO RECURSIVA ---
    
    # Buffer para volume
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    # Histórico para o ciclo (inicializado com os últimos resíduos REAIS do treino)
    hist_resid_cycle = list(last_resids_cycle)
    
    preds_vol = []
    preds_vazao_hibrida = [] 
    
    for i in range(len(df_test)):
        date = df_test.index[i]
        
        # 1. CALCULAR VAZÃO
        
        # A) Pega a Base (Climatologia)
        val_base_clim = get_clim_feature(pd.DatetimeIndex([date]), curva_clim)[0]
        
        # B) Prevê o Ciclo (Usando o ML treinado nos resíduos do Fourier)
        lags_cycle = np.array(hist_resid_cycle[-LAGS_CICLO:][::-1])
        cycle_pred = model_ml_ciclo.predict(pd.DataFrame([lags_cycle], columns=[f'lag_{k}' for k in range(1,LAGS_CICLO+1)]))[0]
        
        # C) SOMA HÍBRIDA: Climatologia + Ciclo(Fourier)
        if cycle_pred > 0:
            if cycle_pred - hist_resid_cycle[-1] > 0:
                cycle_pred = cycle_pred * 0.2  # Ajuste de escala do ciclo
            else:
                cycle_pred = cycle_pred * 0.8  # Ajuste de escala do ciclo
        vazao_final = val_base_clim + cycle_pred
        
        if vazao_final < 0: vazao_final = 0
        preds_vazao_hibrida.append(vazao_final)
        
        # Atualiza histórico de ciclo com o previsto (Recursão)
        hist_resid_cycle.append(cycle_pred)
        
        # 2. CALCULAR VOLUME
        vol_t1 = buffer_vols[-1]
        vol_diff = vol_t1 - buffer_vols[-2]
        
        # Cria DF com dados brutos
        input_vol = pd.DataFrame({
            'vol_lag1': [vol_t1],
            'vol_diff': [vol_diff],
            'vazao_input': [vazao_final]
        })
        
        # --- APLICA SCALING ---
        # Usa o scaler treinado para transformar os dados brutos na escala correta
        input_vol_scaled = scaler_vol.transform(input_vol)
        
        # Prevê
        pred_vol = model_vol.predict(input_vol_scaled)[0]
        preds_vol.append(pred_vol)
        
        buffer_vols.append(pred_vol)
        buffer_vols.pop(0)

    # --- AVALIAÇÃO ---
    rmse_vol = np.sqrt(mean_squared_error(df_test[nome_coluna_volume], preds_vol))
    rmse_vazao = np.sqrt(mean_squared_error(df_test[nome_coluna_vazao_natural], preds_vazao_hibrida))
    
    resultados_vol.append(rmse_vol)
    
    print(f"Fold {fold+1} | RMSE Volume: {rmse_vol:.4f} | RMSE Vazão: {rmse_vazao:.4f}")
    
    # PLOT
    fig, ax = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    
    # Gráfico Volume
    ax[0].plot(df_test.index, df_test[nome_coluna_volume], label='Volume Real', color='navy')
    ax[0].plot(df_test.index, preds_vol, label='Volume Previsto', color='darkorange', linestyle='--')
    ax[0].set_title(f"Fold {fold+1} - Volume")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)
    
    # Gráfico Vazão (O Pulo do Gato)
    ax[1].plot(df_test.index, df_test[nome_coluna_vazao_natural], label='Real', color='black', alpha=0.3)
    
    # Mostra a base Climatológica
    clim_base = get_clim_feature(df_test.index, curva_clim)
    ax[1].plot(df_test.index, clim_base, label='Base (Climatologia)', color='green', linestyle=':', linewidth=1)
    
    # Mostra o resultado final
    ax[1].plot(df_test.index, preds_vazao_hibrida, label='Final (Clim + Ciclo ML)', color='red', linewidth=1.5)
    
    ax[1].set_title("Composição da Vazão: Base Climatológica + Ciclo Aprendido via Fourier")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio (Volume): {np.mean(resultados_vol):.4f}")

### FIM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler # <--- NOVO IMPORT
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
VALIDATION_SIZE = 90
LAGS_CICLO = 2  # Lags do resíduo da vazão
LAGS_VOLUME = 1

# Supondo df já carregado no ambiente
# df = ... 

# ==============================================================================
# 1. CLIMATOLOGIA (A BASE DE TUDO)
# ==============================================================================
def calcular_climatologia_robusta(df_train, col_target, window_days=7):
    temp = df_train[[col_target]].copy()
    temp['day_of_year'] = temp.index.dayofyear
    climatologia = {}
    for day in range(1, 367):
        d_min = day - (window_days // 2)
        d_max = day + (window_days // 2)
        mask = (temp['day_of_year'] >= d_min) & (temp['day_of_year'] <= d_max)
        vals = temp.loc[mask, col_target]
        if len(vals) > 0:
            # Remove outliers extremos (10% - 90%)
            vals = vals[(vals >= vals.quantile(0.10)) & (vals <= vals.quantile(0.90))]
            media = vals.mean() if len(vals) > 0 else 0
        else:
            media = 0
        climatologia[day] = media
    return pd.Series(climatologia, name='clim_base')

def get_clim_feature(index, curva_clim):
    """Retorna array com valores da climatologia para as datas do index"""
    days = index.dayofyear
    return np.array([curva_clim.get(d, curva_clim.iloc[-1]) for d in days])

# ==============================================================================
# 2. MODELO MATEMÁTICO (PARA EXTRAIR O CICLO DA VAZÃO)
# ==============================================================================
def criar_feat_fourier(index):
    dp = DeterministicProcess(
        index=index, 
        constant=True, 
        order=1, 
        seasonal=False, 
        additional_terms=[CalendarFourier("YE", 2)], 
        drop=True
    )
    return dp.in_sample()

def treinar_modelo_extracao_residuo(df_train, col_target):
    y = df_train[col_target]
    X = criar_feat_fourier(df_train.index)
    
    model_math = Ridge(alpha = 0.5)
    model_math.fit(X, y)
    
    # Curva matemática limpa
    y_math_clean = pd.Series(model_math.predict(X), index=y.index)
    
    # Resíduo (Ciclo)
    residuo_ciclo = y - y_math_clean
    return model_math, residuo_ciclo

# ==============================================================================
# 3. MODELO ML (APRENDE O PADRÃO DO CICLO DA VAZÃO)
# ==============================================================================
def criar_lags(series, lags):
    df_lags = pd.DataFrame(index=series.index)
    for l in range(1, lags+1):
        df_lags[f'lag_{l}'] = series.shift(l)
    return df_lags

def treinar_modelo_ciclo_ml(residuo_series):
    X_lags = criar_lags(residuo_series, LAGS_CICLO).dropna()
    y_target = residuo_series.loc[X_lags.index]
    
    # KNN captura padrões locais de repetição do erro
    model_ml = KNeighborsRegressor(n_neighbors=5)
    model_ml.fit(X_lags, y_target)
    
    last_resids = residuo_series.tail(LAGS_CICLO).values
    return model_ml, last_resids

# ==============================================================================
# 4. MODELO DE VOLUME (COM SCALING E CLIMATOLOGIA)
# ==============================================================================
def treinar_modelo_volume(df_train, col_vol, col_vn, curva_clim_vol):
    y = df_train[col_vol]
    X = pd.DataFrame(index=df_train.index)
    
    # Features Autoregressivas
    X['vol_lag1'] = df_train[col_vol].shift(1)
    X['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # Feature Exógena 1: Vazão Real
    X['vazao_input'] = df_train[col_vn] 
    
    # Feature Exógena 2: Climatologia do Volume (O ÂNCORA)
    X['vol_clim'] = get_clim_feature(df_train.index, curva_clim_vol)
    
    X_full = X.dropna()
    y_train = y.loc[X_full.index]
    
    # --- APLICAÇÃO DO SCALER ---
    scaler = StandardScaler()
    
    # Fit e Transform nos dados de treino
    # O scaler aprende Média e Desvio Padrão aqui
    X_scaled_array = scaler.fit_transform(X_full)
    
    # Transformamos de volta para DF apenas para manter nomes das colunas (opcional, mas bom para debug)
    X_scaled = pd.DataFrame(X_scaled_array, columns=X_full.columns, index=X_full.index)
    
    model_vol = LinearRegression()
    model_vol.fit(X_scaled, y_train)
    
    # Print dos coeficientes (AGORA SÃO COMPARÁVEIS GRAÇAS AO SCALING)
    print(f"\n--- Pesos Normalizados do Modelo de Volume ---")
    coefs = pd.Series(model_vol.coef_, index=X_full.columns)
    print(coefs.sort_values(ascending=False))
    print("-" * 30)
    
    # Retornamos o MODELO e o SCALER treinado
    return model_vol, scaler

# ==============================================================================
# 5. LOOP DE VALIDAÇÃO
# ==============================================================================
tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)
resultados_vol = []

print("=== INICIANDO MODELO FINAL: Vol(Lags + VazãoHybrid + VolClim) COM SCALING ===\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
    
    # --- A. TREINAMENTO ---
    
    # 1. Calcular Climatologias (Vazão e Volume)
    curva_clim_vazao = calcular_climatologia_robusta(df_train, nome_coluna_vazao_natural)
    curva_clim_volume = calcular_climatologia_robusta(df_train, nome_coluna_volume)
    
    # 2. Pipeline Vazão
    model_math, serie_residuos_treino = treinar_modelo_extracao_residuo(df_train, nome_coluna_vazao_natural)
    model_ml_ciclo, last_resids_cycle = treinar_modelo_ciclo_ml(serie_residuos_treino)
    
    # 3. Treino do Volume (Retorna Modelo + Scaler)
    model_vol, scaler_vol = treinar_modelo_volume(df_train, nome_coluna_volume, nome_coluna_vazao_natural, curva_clim_volume)
    
    # --- B. PREVISÃO RECURSIVA ---
    
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    hist_resid_cycle = list(last_resids_cycle)
    
    preds_vol = []
    
    for i in range(len(df_test)):
        date = df_test.index[i]
        
        # --- PASSO 1: CALCULAR VAZÃO HÍBRIDA ---
        val_base_vazao = get_clim_feature(pd.DatetimeIndex([date]), curva_clim_vazao)[0]
        
        lags_cycle = np.array(hist_resid_cycle[-LAGS_CICLO:][::-1])
        cycle_pred = model_ml_ciclo.predict(pd.DataFrame([lags_cycle], columns=[f'lag_{k}' for k in range(1,LAGS_CICLO+1)]))[0]
        
        vazao_final = val_base_vazao + cycle_pred
        if vazao_final < 0: vazao_final = 0
        
        hist_resid_cycle.append(cycle_pred)
        
        # --- PASSO 2: CALCULAR VOLUME ---
        vol_t1 = buffer_vols[-1]
        vol_diff = vol_t1 - buffer_vols[-2]
        vol_clim_hoje = get_clim_feature(pd.DatetimeIndex([date]), curva_clim_volume)[0]
        
        # Monta o DataFrame bruto
        input_vol = pd.DataFrame({
            'vol_lag1': [vol_t1],
            'vol_diff': [vol_diff],
            'vazao_input': [vazao_final],
            'vol_clim': [vol_clim_hoje]
        })
        
        # --- APLICAÇÃO DO SCALER (CRUCIAL) ---
        # Transformamos os dados brutos usando a mesma escala aprendida no treino
        input_vol_scaled = scaler_vol.transform(input_vol)
        
        # Prevê usando dados escalados
        pred_vol = model_vol.predict(input_vol_scaled)[0]
        preds_vol.append(pred_vol)
        
        buffer_vols.append(pred_vol)
        buffer_vols.pop(0)

    # --- AVALIAÇÃO ---
    rmse_vol = np.sqrt(mean_squared_error(df_test[nome_coluna_volume], preds_vol))
    resultados_vol.append(rmse_vol)
    
    print(f"Fold {fold+1} | RMSE Volume: {rmse_vol:.4f}")
    
    # PLOT
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(df_test.index, df_test[nome_coluna_volume], label='Real', color='navy', linewidth=2)
    ax.plot(df_test.index, preds_vol, label='Previsto', color='darkorange', linestyle='--', linewidth=2)
    
    vol_clim_test = get_clim_feature(df_test.index, curva_clim_volume)
    ax.plot(df_test.index, vol_clim_test, label='Climatologia (Âncora)', color='green', linestyle=':', alpha=0.6)
    
    ax.set_title(f"Fold {fold+1} - Previsão Volume (RMSE: {rmse_vol:.2f})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print(f"\nRMSE Médio (Volume): {np.mean(resultados_vol):.4f}")

# Teste do Oraculo -> Usar vazao Real apenas pra ver se ela influencia

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.metrics import mean_squared_error
import warnings
import matplotlib.pyplot as plt

# Ignore warnings
warnings.filterwarnings('ignore', message='.*deprecated.*', category=DeprecationWarning)

# --- 1. CONFIGURAÇÃO E DADOS ---
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df[(df['Data']<='2014-03-01')|(df['Data']>='2016-02-15')]
df.set_index("Data", inplace=True)

VALIDATION_SIZE = 90
# LAGS_VAZAO não é mais necessário aqui pois usaremos a REAL
LAGS_VOLUME = 1

# --- 2. FUNÇÕES AUXILIARES ---

def criar_features_deterministicas_volume(index):
    dp = DeterministicProcess(
        index=index,
        constant=True, order=1,
        additional_terms=[
            Fourier(period=365.25 * 2, order=1),
            CalendarFourier(freq="YE", order=2)
        ],
        drop=True
    )
    return dp.in_sample()

def treinar_modelo_volume(df_train, col_vol, col_vn, col_vj, model_vol):
    y = df_train[col_vol]
    
    X_dynamic = pd.DataFrame(index=df_train.index)
    X_dynamic['vol_lag1'] = df_train[col_vol].shift(1)
    
    # 1ª Derivada
    X_dynamic['vol_diff'] = df_train[col_vol].shift(1) - df_train[col_vol].shift(2)
    
    # 2ª Derivada
    X_dynamic['vol_diff2'] = (df_train[col_vol].shift(1) - df_train[col_vol].shift(2)) - \
                             (df_train[col_vol].shift(2) - df_train[col_vol].shift(3))

    X_dynamic['vazao_n'] = df_train[col_vn] 
    X_dynamic['vazao_j'] = df_train[col_vj] 
    
    X_trend = criar_features_deterministicas_volume(df_train.index)
    
    X_full = pd.concat([X_dynamic, X_trend], axis=1).dropna()
    y_train = y.loc[X_full.index]
    
    model_vol.fit(X_full, y_train)
    return model_vol, X_full

def mostrar_equacao_linear(modelo, nomes_colunas):
    coeficientes = modelo.coef_
    intercepto = modelo.intercept_
    
    df_params = pd.DataFrame({
        'Variável': nomes_colunas,
        'Peso (Coeficiente)': coeficientes
    })
    
    df_params['Impacto Absoluto'] = df_params['Peso (Coeficiente)'].abs()
    df_params = df_params.sort_values(by='Impacto Absoluto', ascending=False).drop(columns='Impacto Absoluto')
    
    print("=== EQUAÇÃO DO MODELO LINEAR ===")
    print(f"Intercepto (Valor Base): {intercepto:.4f}")
    print(df_params)
    print("-" * 30)

# --- 3. LOOP DE VALIDAÇÃO (ORACLE TEST) ---

tscv = TimeSeriesSplit(n_splits=5, test_size=VALIDATION_SIZE)

resultados = []

print("Iniciando Validação Cruzada (COM VAZÃO REAL/ORACLE)...\n")

for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
    # Separação dos dados
    df_train = df.iloc[train_idx]
    df_test = df.iloc[test_idx]
    
    print(f"Fold {fold+1}: Treino até {df_train.index[-1].date()} | Teste de {df_test.index[0].date()} a {df_test.index[-1].date()}")

    # =================================================================
    # ETAPA 1: TREINAMENTO (Apenas Volume)
    # =================================================================
    
    # NÃO TREINAMOS MAIS VAZÃO AQUI. ASSUMIMOS QUE JÁ TEMOS ELA.
    
    model_volume = LinearRegression()
    model_volume, X_full_eq = treinar_modelo_volume(
        df_train, nome_coluna_volume, nome_coluna_vazao_natural, nome_coluna_vazao_jusante, model_volume
    )
    
    if fold == 0: # Mostrar equação só no primeiro fold pra não poluir
        mostrar_equacao_linear(model_volume, X_full_eq.columns)

    # =================================================================
    # ETAPA 2: PREVISÃO RECURSIVA (Volume previsto, Vazão Real)
    # =================================================================
    
    X_trend_test_v = criar_features_deterministicas_volume(df_test.index)
    
    # Buffer para as derivadas (ainda precisamos do histórico de volume)
    buffer_vols = list(df_train[nome_coluna_volume].tail(3).values)
    
    preds_volume = []
    
    # LOOP DIA A DIA
    for i in range(len(df_test)):
        current_date = df_test.index[i]
        idx_atual = [df_test.index[i]]
        
        # --- MÁGICA: PEGAR VAZÃO REAL DO FUTURO (TESTE DE SANIDADE) ---
        # Como estamos no "df_test", temos o valor real na linha 'i'
        vn_real = df_test.iloc[i][nome_coluna_vazao_natural]
        vj_real = df_test.iloc[i][nome_coluna_vazao_jusante]
        
        # --- Features do Volume ---
        # Recupera valores do buffer (Volume ainda é recursivo, pois não sabemos o volume de amanhã)
        vol_t_1 = buffer_vols[-1]
        vol_t_2 = buffer_vols[-2]
        vol_t_3 = buffer_vols[-3]
        
        # Derivadas
        feat_diff = vol_t_1 - vol_t_2
        feat_diff2 = (vol_t_1 - vol_t_2) - (vol_t_2 - vol_t_3)
        
        df_dynamic = pd.DataFrame({
            'vol_lag1':  [vol_t_1],
            'vol_diff':  [feat_diff], 
            'vol_diff2': [feat_diff2],
            'vazao_n':   [vn_real], # <--- AQUI ENTRA O VALOR REAL
            'vazao_j':   [vj_real]  # <--- AQUI ENTRA O VALOR REAL
        }, index=idx_atual)
        
        # Parte Estática
        df_static = X_trend_test_v.iloc[[i]] 
        
        # Concatenar
        X_vol_input = pd.concat([df_dynamic, df_static], axis=1)
        
        # Prever Volume
        vol_pred = model_volume.predict(X_vol_input)[0]
        preds_volume.append(vol_pred)
        
        # Atualiza buffer
        buffer_vols.append(vol_pred) 
        buffer_vols.pop(0)

    # =================================================================
    # ETAPA 3: AVALIAÇÃO
    # =================================================================
    y_true = df_test[nome_coluna_volume]
    rmse = np.sqrt(mean_squared_error(y_true, preds_volume))
    mae = np.mean(np.abs(y_true - preds_volume))
    resultados.append(rmse)
    
    print(f"RMSE Fold {fold+1} (Com Vazão Real): {rmse:.4f}")
    print("-" * 30)
    
    plt.figure(figsize=(12, 5))
    plt.plot(y_true.index, y_true, label='Real', color='navy', linewidth=2)
    plt.plot(y_true.index, preds_volume, label='Simulação (Vazão Perfeita)', color='green', linestyle='--', linewidth=2)
    plt.title(f'Fold {fold+1} | RMSE: {rmse:.2f}')
    plt.xlabel('Data')
    plt.ylabel(nome_coluna_volume)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("-" * 30)

print(f"\nRMSE Médio (Ideal): {np.mean(resultados):.4f}")

In [ ]:

### VAZAO APARENTE ESTAR SUPERESTIMADA PELO NOSSO MODELO.

## CALCULAR A VAZAO HOJE. -> Além do lag da vazao e sazonalidade e tendencia
## Usar um minimo historico da vazao.
## hoje é dia 10/12/2025 -> Pego do dia 8/12/2020(21,22,23,24) até dia 12/12/2020(21,22,23,24) e faço a media 
## E incluo como variavel

index = ['2024-12-05', '2024-12-06', '2024-12-07', '2024-12-08', '2024-12-09',
         '2023-12-05', '2023-12-06', '2023-12-07', '2023-12-08', '2023-12-09',
         '2022-12-05', '2022-12-06', '2022-12-07', '2022-12-08', '2022-12-09',
         '2021-12-05', '2021-12-06', '2021-12-07', '2021-12-08', '2021-12-09'
         ]


import pandas as pd

# Índices fornecidos
index = [
    '2024-12-05', '2024-12-06', '2024-12-07', '2024-12-08', '2024-12-09',
    '2023-12-05', '2023-12-06', '2023-12-07', '2023-12-08', '2023-12-09',
    '2022-12-05', '2022-12-06', '2022-12-07', '2022-12-08', '2022-12-09',
    '2021-12-05', '2021-12-06', '2021-12-07', '2021-12-08', '2021-12-09'
]

# Seleciona os valores
vals = df.loc[index, 'Vazão Jusante (m³/s)']

# Calcula limites pelos quantis
q_low = vals.quantile(0.05)   # limite inferior
q_high = vals.quantile(0.95)  # limite superior

# Remove outliers
vals_filtered = vals[(vals >= q_low) & (vals <= q_high)]

# Calcula o mínimo sem outliers
min_val = vals_filtered.min()

print("Mínimo sem outliers:", min_val)
df['Vazão Jusante (m³/s)'][index].min(), df['Vazão Jusante (m³/s)'][index].mean(), df['Vazão Jusante (m³/s)'][index].max()